<a href="https://colab.research.google.com/github/Md-Sanzid-Bin-Hossain/Multi-modal-IMU-and-Wearable-Camera-Dataset/blob/main/Wearable_Motion_capture_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1 — Setup: Install dependencies, mount Google Drive, load dataset

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "h5py", "pingouin", "-q"], check=True)

# ── Core imports ──────────────────────────────────────────────────────────────
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from itertools import combinations
from collections import OrderedDict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from io import StringIO
import os
import warnings
warnings.filterwarnings("ignore")

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── HDF5 path — update if stored in a subfolder ───────────────────────────────
HDF5_PATH = "/content/drive/MyDrive/WMCG_dataset.h5"

# ── Verify ────────────────────────────────────────────────────────────────────
assert os.path.isfile(HDF5_PATH), f"❌ File not found: {HDF5_PATH}"
file_gb = os.path.getsize(HDF5_PATH) / (1024**3)
print(f"✅ Dataset found: {HDF5_PATH}")
print(f"   Size: {file_gb:.2f} GB")

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = "/content/gait_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"   Output dir: {OUTPUT_DIR}")

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.0)
COLORS   = sns.color_palette("tab10")
P_COLORS = plt.cm.tab10(np.linspace(0, 1, 10))
plt.rcParams["figure.dpi"] = 120

MessageError: Error: credential propagation was unsuccessful

# Cell 2 — Constants: Participants, joints, conditions, speed labels

In [ ]:
# ── Participants ──────────────────────────────────────────────────────────────
PARTICIPANTS = [f"{i:02d}" for i in range(1, 11)]
P_LABELS     = [f"P{i:02d}" for i in range(1, 11)]
N_POINTS     = 101  # gait cycle normalization points

# ── Joints ────────────────────────────────────────────────────────────────────
JOINT_ORDER  = ["Hip Flexion", "Knee", "Ankle"]

JOINT_PAIRS = [
    ("hip_flexion_r", "hip_flexion_l", "Hip Flexion"),
    ("knee_angle_r",  "knee_angle_l",  "Knee"),
    ("ankle_angle_r", "ankle_angle_l", "Ankle"),
]

STAT_JOINTS_R = OrderedDict([
    ("hip_flexion_r", "Hip Flexion"),
    ("knee_angle_r",  "Knee"),
    ("ankle_angle_r", "Ankle"),
])
STAT_JOINTS_L = OrderedDict([
    ("hip_flexion_l", "Hip Flexion"),
    ("knee_angle_l",  "Knee"),
    ("ankle_angle_l", "Ankle"),
])
R_TO_L = {
    "hip_flexion_r": "hip_flexion_l",
    "knee_angle_r":  "knee_angle_l",
    "ankle_angle_r": "ankle_angle_l",
}

JOINT_SIDE_ORDER = [
    "Hip Flexion (R)", "Hip Flexion (L)",
    "Knee (R)",        "Knee (L)",
    "Ankle (R)",       "Ankle (L)",
]

# ── Conditions ────────────────────────────────────────────────────────────────
SPEED_ORDER     = ["slow", "normal", "fast", "vfast"]

MODE_ORDER_FULL = [
    "Treadmill", "Overground",
    "Slope Ascent", "Slope Descent",
    "Stair Ascent", "Stair Descent",
]

ICC_COL_ORDER = [
    "Slope Ascent", "Slope Descent",
    "Stair Ascent", "Stair Descent",
]

# ── Speed labels for plots ────────────────────────────────────────────────────
TM_SPEEDS_LABEL = {
    "slow": "0.73", "normal": "1.11",
    "fast": "1.49", "vfast":  "1.87",
}
OG_SPEEDS_LABEL = {
    "slow": "0.68", "normal": "0.80",
    "fast": "1.02", "vfast":  "1.21",
}
TM_SPEEDS_MS = {
    "slow": 0.73, "normal": 1.11,
    "fast": 1.49, "vfast":  1.87,
}
OG_SPEEDS_MS = {
    "slow": 0.67, "normal": 0.79,
    "fast": 0.99, "vfast":  1.16,
}

# ── Froude numbers (for treadmill speed computation) ─────────────────────────
g      = 9.81
FROUDE = {
    "slow": 0.2689, "normal": 0.4089,
    "fast": 0.5489, "vfast":  0.6889,
}

# ── Demographics (leg lengths needed for Froude speed computation) ────────────
LEG_LENGTHS = {
    "P01": 0.75, "P02": 0.71, "P03": 0.73, "P04": 0.70,
    "P05": 0.81, "P06": 0.77, "P07": 0.70, "P08": 0.80,
    "P09": 0.77, "P10": 0.77,
}

print("✅ Constants loaded.")
print(f"   Participants : {len(PARTICIPANTS)}")
print(f"   Joints       : {JOINT_ORDER}")
print(f"   Speed conds  : {SPEED_ORDER}")
print(f"   Modes        : {MODE_ORDER_FULL}")

# Cell 3 — Dataset Overview: Demographics, conditions, modality summary

In [ ]:
# ── Load demographics from HDF5 ───────────────────────────────────────────────
with h5py.File(HDF5_PATH, "r") as hf:
    pids     = [p.decode() for p in hf["metadata/demographics/participant_ids"][:]]
    ages     = hf["metadata/demographics/age"][:]
    weights  = hf["metadata/demographics/weight"][:]
    heights  = hf["metadata/demographics/height"][:]
    leg_lens = hf["metadata/demographics/leg_length"][:]
    genders  = [g.decode() for g in hf["metadata/demographics/gender"][:]]
    tm_speeds = {
        cond: hf[f"metadata/treadmill_speeds_ms/{cond}"][:]
        for cond in SPEED_ORDER
    }

# ── Demographics table ────────────────────────────────────────────────────────
demo_df = pd.DataFrame({
    "Subject":        pids,
    "Gender":         genders,
    "Age (yrs)":      ages.astype(int),
    "Weight (kg)":    weights.round(1),
    "Height (m)":     heights.round(2),
    "Leg Length (m)": leg_lens.round(2),
})

print("="*60)
print("PARTICIPANT DEMOGRAPHICS")
print("="*60)
print(demo_df.to_string(index=False))
print("-"*60)
n_male   = genders.count("Male")
n_female = genders.count("Female")
print(f"  Age    : {ages.mean():.1f} ± {ages.std(ddof=1):.1f} years")
print(f"  Weight : {weights.mean():.1f} ± {weights.std(ddof=1):.1f} kg")
print(f"  Height : {heights.mean():.2f} ± {heights.std(ddof=1):.2f} m")
print(f"  Gender : {n_male} Male, {n_female} Female")

# ── Treadmill speeds per subject ──────────────────────────────────────────────
print("\n" + "="*60)
print("TREADMILL SPEEDS PER SUBJECT (m/s, Froude-based)")
print("="*60)
print(f"{'Subject':>8} | {'Slow':>6} {'Normal':>7} {'Fast':>6} {'VFast':>7}")
print("-"*38)
for i, pid in enumerate(pids):
    print(f"  {pid:>6} | "
          f"{tm_speeds['slow'][i]:>6.3f} "
          f"{tm_speeds['normal'][i]:>7.3f} "
          f"{tm_speeds['fast'][i]:>6.3f} "
          f"{tm_speeds['vfast'][i]:>7.3f}")
print("-"*38)
for cond in SPEED_ORDER:
    v = tm_speeds[cond]
    print(f"{'Mean±SD':>8}   "
          f"{np.mean(v):.2f}±{np.std(v,ddof=1):.2f}", end="   ")
print()

# ── Available conditions per participant ──────────────────────────────────────
print("\n" + "="*60)
print("CONDITIONS AVAILABLE IN DATASET")
print("="*60)
condition_info = {
    "Treadmill":  {"conds": ["slow","normal","fast","vfast"],
                   "note":  "Froude-based speed, 4 levels"},
    "Overground": {"conds": ["slow","normal","fast","vfast",
                             "round","obstacles"],
                   "note":  "Self-selected speed"},
    "Slope":      {"conds": ["1","2"],
                   "note":  "20% grade, 2 reps, ascent+descent"},
    "Stair":      {"conds": ["1","2"],
                   "note":  "2 reps, ascent+descent"},
}
with h5py.File(HDF5_PATH, "r") as hf:
    for loco, info in condition_info.items():
        print(f"\n  {loco} — {info['note']}")
        for cond in info["conds"]:
            base    = f"participants/P01/{loco.lower()}/{cond}"
            has_kin = f"{base}/kinematics" in hf
            has_imu = f"{base}/imu"        in hf
            has_hof = f"{base}/hof"        in hf
            has_mrk = f"{base}/markers"    in hf
            if has_kin:
                n_frames = hf[f"{base}/kinematics/data"].shape[0]
                print(f"    {cond:<12} "
                      f"{'Markers':>8}={'✅' if has_mrk else '❌'}  "
                      f"{'Kin':>4}={'✅' if has_kin else '❌'}  "
                      f"{'IMU':>4}={'✅' if has_imu else '❌'}  "
                      f"{'HOF':>4}={'✅' if has_hof else '❌'}  "
                      f"({n_frames} frames @ 100 Hz)")

# ── Modality summary ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("MODALITY SUMMARY")
print("="*60)
modalities = [
    ("Motion capture markers", "100 Hz, Vicon, 34 reflective markers, units: mm"),
    ("OpenSim kinematics",     "100 Hz, joint angles, units: degrees"),
    ("IMU",                    "100 Hz, 8 sensors × 6 ch (ACC+GYRO)"),
    ("HOF features",           "30 Hz, shoe-mounted cameras (left + right)"),
    ("Video",                  "Not included — stored separately (too large)"),
]
for name, desc in modalities:
    icon = "✅" if "Not" not in desc else "⚠️ "
    print(f"  {icon} {name:<28} {desc}")

# Cell 4 — Functions: Heel strike detection, gait cycle extraction, turn filtering, ascent/descent classification

In [ ]:
# ── 4.1 Load raw data from HDF5 ──────────────────────────────────────────────

def load_markers(plabel, loco, cond):
    """
    Load raw marker data from HDF5.
    Returns: data (T × n_markers*3), marker_names, time
    Layout: XYZ interleaved per marker — marker i = cols [i*3, i*3+1, i*3+2]
    """
    base = f"participants/{plabel}/{loco.lower()}/{cond}/markers"
    with h5py.File(HDF5_PATH, "r") as hf:
        if base not in hf:
            return None, None, None
        data         = hf[f"{base}/data"][:]
        marker_names = [m.decode() for m in hf[f"{base}/marker_names"][:]]
        time         = hf[f"{base}/time"][:]
    return data, marker_names, time


def load_kinematics(plabel, loco, cond):
    """
    Load raw kinematics from HDF5.
    Returns: data (T × n_joints), joint_names, time
    Units: degrees
    """
    base = f"participants/{plabel}/{loco.lower()}/{cond}/kinematics"
    with h5py.File(HDF5_PATH, "r") as hf:
        if base not in hf:
            return None, None, None
        data        = hf[f"{base}/data"][:]
        joint_names = [j.decode() for j in hf[f"{base}/joint_names"][:]]
        time        = hf[f"{base}/time"][:]
    return data, joint_names, time


def get_marker_col(data, marker_names, marker, axis):
    """
    Extract a single marker axis from the marker data matrix.
    axis: 'X'=0, 'Y'=1, 'Z'=2
    """
    ax = {"X": 0, "Y": 1, "Z": 2}[axis.upper()]
    idx = marker_names.index(marker)
    return data[:, idx * 3 + ax]


def get_joint_col(data, joint_names, joint):
    """Extract a single joint angle time series."""
    idx = joint_names.index(joint)
    return data[:, idx]


# ── 4.2 Heel strike detection ─────────────────────────────────────────────────

def detect_heel_strikes(heel_y, time, min_dist_sec=0.4, fs=100):
    """
    Detect heel strikes as local minima of heel marker vertical (Y) position.
    Returns array of frame indices.

    Parameters
    ----------
    heel_y       : array (T,) — vertical heel marker position in mm
    time         : array (T,) — time vector in seconds
    min_dist_sec : minimum time between heel strikes (seconds)
    fs           : sampling frequency (Hz)
    """
    min_dist = int(min_dist_sec * fs)
    peaks, _ = find_peaks(-heel_y, distance=min_dist, prominence=30)
    return peaks


# ── 4.3 Gait cycle normalization ──────────────────────────────────────────────

def normalize_cycle(signal, start, end, n_points=101):
    """
    Interpolate one gait cycle to n_points using cubic interpolation.
    Returns normalized cycle or None if segment is too short/has NaNs.
    """
    segment = signal[start:end].astype(float)
    if len(segment) < 10 or np.any(np.isnan(segment)):
        return None
    x_orig = np.linspace(0, 100, len(segment))
    x_norm = np.linspace(0, 100, n_points)
    return interp1d(x_orig, segment, kind='cubic')(x_norm)


def extract_cycles(signal, hs_indices, n_points=101,
                   min_frames=30, max_frames=300):
    """
    Extract all valid gait cycles from a signal using heel strike indices.
    Returns array of shape (n_cycles, n_points) or None.

    Parameters
    ----------
    signal     : array (T,) — any continuous signal (joint angle, etc.)
    hs_indices : array — heel strike frame indices
    min_frames : minimum cycle duration in frames (30 = 0.3s at 100Hz)
    max_frames : maximum cycle duration in frames (300 = 3.0s at 100Hz)
    """
    cycles = []
    for i in range(len(hs_indices) - 1):
        start    = hs_indices[i]
        end      = hs_indices[i + 1]
        duration = end - start
        if min_frames <= duration <= max_frames:
            norm = normalize_cycle(signal, start, end, n_points)
            if norm is not None:
                cycles.append(norm)
    return np.array(cycles) if cycles else None


# ── 4.4 Turn filtering (overground straight walking) ─────────────────────────

def compute_pelvis_velocity(marker_data, marker_names, fs=100,
                             smooth_window=50):
    """
    Compute smoothed forward pelvis velocity from RASI + LASI midpoint.
    Automatically detects forward axis (X or Z) by range.

    Returns velocity_smooth array (T,) in mm/frame.
    Positive = one direction, negative = other, near-zero = turning.
    """
    rasi_x = get_marker_col(marker_data, marker_names, "RASI", "X")
    lasi_x = get_marker_col(marker_data, marker_names, "LASI", "X")
    rasi_z = get_marker_col(marker_data, marker_names, "RASI", "Z")
    lasi_z = get_marker_col(marker_data, marker_names, "LASI", "Z")

    pelvis_x = (rasi_x + lasi_x) / 2.0
    pelvis_z = (rasi_z + lasi_z) / 2.0

    # Forward axis = larger total range
    forward = pelvis_x if (np.nanmax(pelvis_x) - np.nanmin(pelvis_x) >
                           np.nanmax(pelvis_z) - np.nanmin(pelvis_z)) \
              else pelvis_z

    velocity = np.gradient(forward)
    kernel   = np.ones(smooth_window) / smooth_window
    return np.convolve(velocity, kernel, mode='same')


def filter_straight_cycles(hs_indices, velocity,
                            vel_threshold=0.5, max_sign_change_frac=0.3):
    """
    Keep only gait cycles where participant is walking straight.
    Rejects cycles with:
      - Mean absolute velocity below threshold (stopped or turning)
      - Too many velocity sign changes (mid-turn)

    Returns filtered heel strike indices.
    """
    good = []
    for i in range(len(hs_indices) - 1):
        start = hs_indices[i]
        end   = hs_indices[i + 1]
        if end >= len(velocity):
            continue
        cyc_vel  = velocity[start:end]
        mean_abs = np.mean(np.abs(cyc_vel))
        signs    = np.sign(cyc_vel)
        frac_sc  = np.sum(np.abs(np.diff(signs)) > 0) / len(cyc_vel)
        if mean_abs > vel_threshold and frac_sc < max_sign_change_frac:
            good.append(i)

    if not good:
        return np.array([])
    keep = set()
    for i in good:
        keep.add(hs_indices[i])
        keep.add(hs_indices[i + 1])
    return np.sort(np.array(list(keep)))


# ── 4.5 Ascent/descent classification (slope & stair) ────────────────────────

def classify_ascent_descent(marker_data, marker_names, hs_indices,
                             delta_y_threshold=5.0):
    """
    Classify each gait cycle as ascent or descent using pelvis height change.
    Uses midpoint of RASI_Y + LASI_Y.

    delta_y > +threshold → ascending
    delta_y < -threshold → descending
    |delta_y| < threshold → transition/flat (excluded)

    Returns (ascent_hs, descent_hs) — each is array of heel strike indices
    """
    rasi_y   = get_marker_col(marker_data, marker_names, "RASI", "Y")
    lasi_y   = get_marker_col(marker_data, marker_names, "LASI", "Y")
    pelvis_y = (rasi_y + lasi_y) / 2.0

    asc_cycles, desc_cycles = [], []
    for i in range(len(hs_indices) - 1):
        start   = hs_indices[i]
        end     = hs_indices[i + 1]
        if end >= len(pelvis_y):
            continue
        delta_y = pelvis_y[end] - pelvis_y[start]
        if delta_y > delta_y_threshold:
            asc_cycles.append(i)
        elif delta_y < -delta_y_threshold:
            desc_cycles.append(i)

    def rebuild(indices):
        if not indices:
            return np.array([])
        keep = set()
        for i in indices:
            keep.add(hs_indices[i])
            keep.add(hs_indices[i + 1])
        return np.sort(np.array(list(keep)))

    return rebuild(asc_cycles), rebuild(desc_cycles)


# ── 4.6 Master: get heel strikes for any condition ────────────────────────────

def get_heel_strikes(plabel, loco, cond):
    """
    Load marker data and return heel strike indices for any condition.
    Applies appropriate processing per locomotion type:

      Treadmill  → all cycles (no filtering)
      Overground → turn-filtered (straight cycles only)
      Slope/Stair → ascent + descent classified separately

    Returns dict:
      Treadmill/Overground: {"R": hs_R, "L": hs_L, "time": time}
      Slope/Stair:          {"R": hs_R, "L": hs_L,
                             "asc_R": ..., "asc_L": ...,
                             "desc_R": ..., "desc_L": ...,
                             "time": time}
      None if data not found.
    """
    marker_data, marker_names, time = load_markers(plabel, loco, cond)
    if marker_data is None:
        return None

    rhee_y = get_marker_col(marker_data, marker_names, "RHEE", "Y")
    lhee_y = get_marker_col(marker_data, marker_names, "LHEE", "Y")

    hs_R = detect_heel_strikes(rhee_y, time)
    hs_L = detect_heel_strikes(lhee_y, time)

    if len(hs_R) < 3 or len(hs_L) < 3:
        return None

    if loco == "Treadmill":
        return {"R": hs_R, "L": hs_L, "time": time}

    elif loco == "Overground":
        vel   = compute_pelvis_velocity(marker_data, marker_names)
        hs_R  = filter_straight_cycles(hs_R, vel)
        hs_L  = filter_straight_cycles(hs_L, vel)
        return {"R": hs_R, "L": hs_L, "time": time}

    elif loco in ["Slope", "Stair"]:
        asc_R,  desc_R = classify_ascent_descent(
            marker_data, marker_names, hs_R)
        asc_L,  desc_L = classify_ascent_descent(
            marker_data, marker_names, hs_L)
        return {
            "R":      hs_R,    "L":      hs_L,
            "asc_R":  asc_R,   "asc_L":  asc_L,
            "desc_R": desc_R,  "desc_L": desc_L,
            "time":   time,
        }

    return None


# ── 4.7 ROM extraction ────────────────────────────────────────────────────────

def get_participant_rom(plabel, loco, cond,
                        side_key_R="R", side_key_L="L"):
    """
    Compute mean per-cycle ROM for one participant, one condition.
    Returns dict: {"rom_R": {joint: mean_rom}, "rom_L": {joint: mean_rom}}
    or None if data unavailable.
    """
    hs_result = get_heel_strikes(plabel, loco, cond)
    if hs_result is None:
        return None

    kin_data, joint_names, _ = load_kinematics(plabel, loco, cond)
    if kin_data is None:
        return None

    hs_R = hs_result[side_key_R]
    hs_L = hs_result[side_key_L]
    if len(hs_R) < 3 or len(hs_L) < 3:
        return None

    out = {"rom_R": {}, "rom_L": {}}

    for jcol, jname in STAT_JOINTS_R.items():
        if jcol not in joint_names:
            continue
        signal = get_joint_col(kin_data, joint_names, jcol)
        cycles = extract_cycles(signal, hs_R)
        if cycles is not None and len(cycles) >= 3:
            roms = np.ptp(cycles, axis=1)  # ROM per cycle
            out["rom_R"][jcol] = float(np.mean(roms))

    for jcol, jname in STAT_JOINTS_L.items():
        if jcol not in joint_names:
            continue
        signal = get_joint_col(kin_data, joint_names, jcol)
        cycles = extract_cycles(signal, hs_L)
        if cycles is not None and len(cycles) >= 3:
            roms = np.ptp(cycles, axis=1)
            out["rom_L"][jcol] = float(np.mean(roms))

    return out if (out["rom_R"] or out["rom_L"]) else None


def symmetry_index(rom_left, rom_right):
    """Bilateral symmetry index (%) — sign-convention independent."""
    d = abs(rom_left) + abs(rom_right)
    return (2 * abs(rom_right - rom_left) / d * 100) if d > 0 else np.nan


print("✅ All processing functions loaded.")
print("   Functions available:")
funcs = [
    "load_markers(plabel, loco, cond)",
    "load_kinematics(plabel, loco, cond)",
    "get_marker_col(data, marker_names, marker, axis)",
    "get_joint_col(data, joint_names, joint)",
    "detect_heel_strikes(heel_y, time)",
    "normalize_cycle(signal, start, end)",
    "extract_cycles(signal, hs_indices)",
    "compute_pelvis_velocity(marker_data, marker_names)",
    "filter_straight_cycles(hs_indices, velocity)",
    "classify_ascent_descent(marker_data, marker_names, hs_indices)",
    "get_heel_strikes(plabel, loco, cond)  ← master function",
    "get_participant_rom(plabel, loco, cond)",
    "symmetry_index(rom_left, rom_right)",
]
for f in funcs:
    print(f"   • {f}")

# Cell 5 — Walking Speed Analysis: Per-subject speeds for all conditions

In [ ]:
# ── Load pre-computed speed table from HDF5 ───────────────────────────────────
with h5py.File(HDF5_PATH, "r") as hf:
    speed_df = pd.read_csv(StringIO(
        hf["processed/per_subject_speeds_csv"][()].decode()))

print("✅ Speed table loaded.")
print(f"   Shape: {speed_df.shape} — {len(speed_df)} subjects × "
      f"{len(speed_df.columns)-1} conditions\n")

# ── Display per-subject speed table ──────────────────────────────────────────
all_cols = [
    "TM_slow","TM_normal","TM_fast","TM_vfast",
    "OG_slow","OG_normal","OG_fast","OG_vfast",
    "OG_round","OG_obstacle",
    "Slope_ascent","Slope_descent",
    "Stair_ascent","Stair_descent",
]

print("="*110)
print("PER-SUBJECT WALKING SPEEDS (m/s)")
print("="*110)
print(f"{'Subject':>8} | "
      f"{'TM_Sl':>6} {'TM_No':>6} {'TM_Fa':>6} {'TM_VF':>6} | "
      f"{'OG_Sl':>6} {'OG_No':>6} {'OG_Fa':>6} {'OG_VF':>6} "
      f"{'OG_Ro':>6} {'OG_Ob':>6} | "
      f"{'SlAsc':>6} {'SlDes':>6} | "
      f"{'StAsc':>6} {'StDes':>6}")
print("-"*110)

for _, row in speed_df.iterrows():
    vals = [f"{row[c]:.2f}" if not pd.isna(row[c]) else " N/A"
            for c in all_cols]
    print(f"{row.Subject:>8} | "
          f"{vals[0]:>6} {vals[1]:>6} {vals[2]:>6} {vals[3]:>6} | "
          f"{vals[4]:>6} {vals[5]:>6} {vals[6]:>6} {vals[7]:>6} "
          f"{vals[8]:>6} {vals[9]:>6} | "
          f"{vals[10]:>6} {vals[11]:>6} | "
          f"{vals[12]:>6} {vals[13]:>6}")

print("-"*110)
means = [f"{speed_df[c].dropna().mean():>6.2f}" for c in all_cols]
sds   = [f"{speed_df[c].dropna().std(ddof=1):>6.2f}" for c in all_cols]
print(f"{'Mean':>8} | "
      f"{means[0]} {means[1]} {means[2]} {means[3]} | "
      f"{means[4]} {means[5]} {means[6]} {means[7]} "
      f"{means[8]} {means[9]} | "
      f"{means[10]} {means[11]} | "
      f"{means[12]} {means[13]}")
print(f"{'SD':>8} | "
      f"{sds[0]} {sds[1]} {sds[2]} {sds[3]} | "
      f"{sds[4]} {sds[5]} {sds[6]} {sds[7]} "
      f"{sds[8]} {sds[9]} | "
      f"{sds[10]} {sds[11]} | "
      f"{sds[12]} {sds[13]}")

# ── Speed summary plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Walking Speed Summary — All Conditions (n=10)",
             fontsize=13, fontweight='bold')

# Panel 1 — Treadmill vs Overground straight walking
ax = axes[0]
tm_means = [speed_df[f"TM_{s}"].mean() for s in SPEED_ORDER]
tm_sds   = [speed_df[f"TM_{s}"].std()  for s in SPEED_ORDER]
og_means = [speed_df[f"OG_{s}"].mean() for s in SPEED_ORDER]
og_sds   = [speed_df[f"OG_{s}"].std()  for s in SPEED_ORDER]
x = np.arange(4)
w = 0.32
ax.bar(x - w/2, tm_means, w, yerr=tm_sds, label="Treadmill",
       color=COLORS[0], capsize=4, alpha=0.85, edgecolor='black',
       linewidth=0.7)
ax.bar(x + w/2, og_means, w, yerr=og_sds, label="Overground",
       color=COLORS[1], capsize=4, alpha=0.85, edgecolor='black',
       linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels(["Slow","Normal","Fast","V.Fast"])
ax.set_ylabel("Speed (m/s)")
ax.set_title("Treadmill vs Overground\n(Straight Walking)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 2 — Overground special conditions (Round + Obstacles)
ax = axes[1]
special_labels = ["Round", "Obstacles"]          # ← fixed labels
special_cols   = ["OG_round", "OG_obstacle"]
special_means  = [speed_df[c].mean() for c in special_cols]
special_sds    = [speed_df[c].std()  for c in special_cols]
x2 = np.arange(len(special_labels))
w2 = 0.35                                        # ← slimmer bars
bars = ax.bar(x2, special_means, w2, yerr=special_sds,
              color=[COLORS[2], COLORS[3]], capsize=4,
              alpha=0.85, edgecolor='black', linewidth=0.7)
for bar, m, s in zip(bars, special_means, special_sds):
    ax.text(bar.get_x() + bar.get_width()/2,
            m + s + 0.02, f"{m:.2f}", ha='center',
            fontsize=10, fontweight='bold')
ax.set_xticks(x2)
ax.set_xticklabels(special_labels)
ax.set_ylabel("Speed (m/s)")
ax.set_title("Overground Special\nConditions")
ax.set_xlim(-0.5, 1.5)                           # ← tighter x range
ax.set_ylim(0, 1.2)
ax.grid(True, alpha=0.3, axis='y')

# Panel 3 — Slope and Stair
ax = axes[2]
slope_stair_labels = ["Slope\nAscent","Slope\nDescent",
                       "Stair\nAscent","Stair\nDescent"]
slope_stair_cols   = ["Slope_ascent","Slope_descent",
                       "Stair_ascent","Stair_descent"]
ss_means = [speed_df[c].mean() for c in slope_stair_cols]
ss_sds   = [speed_df[c].std()  for c in slope_stair_cols]
x3 = np.arange(len(slope_stair_labels))
w3 = 0.45
bars = ax.bar(x3, ss_means, w3, yerr=ss_sds,
              color=[COLORS[4], COLORS[5], COLORS[6], COLORS[7]],
              capsize=4, alpha=0.85, edgecolor='black', linewidth=0.7)
for bar, m, s in zip(bars, ss_means, ss_sds):
    ax.text(bar.get_x() + bar.get_width()/2,
            m + s + 0.01, f"{m:.2f}", ha='center',
            fontsize=10, fontweight='bold')
ax.set_xticks(x3)
ax.set_xticklabels(slope_stair_labels)
ax.set_ylabel("Speed (m/s)")
ax.set_title("Slope & Stair\n(Avg Rep1+Rep2)")
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_speed_summary.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_speed_summary.png")

# Cell 6 — Gait Cycle Visualization: Population mean ± SD curves for all conditions

In [ ]:
def plot_population_gait(condition_label, loco, cond,
                         side_key_R="R", side_key_L="L",
                         save_path=None):
    gait_pct = np.linspace(0, 100, N_POINTS)
    fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
    fig.suptitle(condition_label, fontsize=13, fontweight='bold', y=0.98)

    all_curves = {jR: {"R": [], "L": []} for jR, jL, _ in JOINT_PAIRS}
    all_roms   = {jR: {"R": [], "L": []} for jR, jL, _ in JOINT_PAIRS}

    for i, (pid, plabel) in enumerate(zip(PARTICIPANTS, P_LABELS)):
        hs_result = get_heel_strikes(plabel, loco, cond)
        if hs_result is None:
            continue
        hs_R = hs_result[side_key_R]
        hs_L = hs_result[side_key_L]
        if len(hs_R) < 3 or len(hs_L) < 3:
            continue
        kin_data, joint_names, _ = load_kinematics(plabel, loco, cond)
        if kin_data is None:
            continue

        for jR, jL, jname in JOINT_PAIRS:
            for jcol, side, hs in [(jR,"R",hs_R),(jL,"L",hs_L)]:
                if jcol not in joint_names:
                    continue
                signal = get_joint_col(kin_data, joint_names, jcol)
                cycles = extract_cycles(signal, hs)
                if cycles is not None and len(cycles) >= 3:
                    all_curves[jR][side].append(
                        (i, np.mean(cycles, axis=0)))
                    all_roms[jR][side].append(
                        float(np.mean(np.ptp(cycles, axis=1))))

    legend_handles = None
    legend_labels  = None

    for row, (jR, jL, jname) in enumerate(JOINT_PAIRS):
        for col, (side, side_label) in enumerate(
                [("R","Right"), ("L","Left")]):
            ax     = axes[row, col]
            curves = all_curves[jR][side]

            if not curves:
                ax.text(0.5, 0.5, "No data",
                        transform=ax.transAxes,
                        ha='center', color='red', fontsize=11)
                continue

            # Individual participant curves
            for p_idx, curve in curves:
                ax.plot(gait_pct, curve,
                        color=P_COLORS[p_idx],
                        alpha=0.45, linewidth=0.9)

            # Population mean ± SD
            stack    = np.array([c for _, c in curves])
            pop_mean = np.mean(stack, axis=0)
            pop_sd   = np.std(stack,  axis=0)
            n_subj   = len(stack)

            ax.plot(gait_pct, pop_mean,
                    color='black', linewidth=2.2,
                    zorder=5, label=f"Mean (n={n_subj})")
            ax.fill_between(gait_pct,
                            pop_mean - pop_sd,
                            pop_mean + pop_sd,
                            alpha=0.18, color='black',
                            label='±1 SD')
            ax.axvline(60, color='#cc0000', linestyle='--',
                       alpha=0.6, linewidth=1.0,
                       label='Toe-off (60%)')

            # Capture handles once
            if legend_handles is None:
                legend_handles, legend_labels = ax.get_legend_handles_labels()

            if row == 0:
                ax.set_title(f"{side_label} Leg",
                             fontsize=11, fontweight='bold')

            ax.set_ylabel(f"{jname} (°)", fontsize=9)
            if row == 2:
                ax.set_xlabel("Gait Cycle (%)", fontsize=9)

            ax.grid(True, alpha=0.22, linestyle='--')
            ax.set_xlim(0, 100)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)

# ── Single shared legend centered at top — no overlap possible ────────────
    if legend_handles:
        fig.legend(legend_handles, legend_labels,
                   loc='upper center',
                   bbox_to_anchor=(0.5, 0.955),  # ← lower than before
                   ncol=3,
                   fontsize=8.5,
                   framealpha=0.92,
                   edgecolor='#aaaaaa',
                   handlelength=1.6,
                   columnspacing=1.5)

    plt.tight_layout(rect=[0, 0, 1, 0.90])       # ← more space at top
    plt.subplots_adjust(hspace=0.18, wspace=0.28)



    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✅ Saved: {os.path.basename(save_path)}")
    plt.show()
    plt.close()

    # ── ROM summary table printed below figure ─────────────────────────────────
    print(f"\n  ROM Summary — {condition_label}")
    print(f"  {'Joint':<15} {'Side':<6} {'Mean (°)':>9} {'SD (°)':>8} {'N':>4}")
    print(f"  {'-'*46}")
    for jR, jL, jname in JOINT_PAIRS:
        for side, jcol in [("R", jR), ("L", jL)]:
            roms = all_roms[jR][side]
            if roms:
                print(f"  {jname:<15} {side:<6} "
                      f"{np.mean(roms):>9.1f} "
                      f"{np.std(roms):>8.1f} "
                      f"{len(roms):>4}")


# ── Run for all conditions ────────────────────────────────────────────────────

# Treadmill — 4 speeds
print("="*60)
print("TREADMILL — Population Gait Cycles")
print("="*60)
for spd in SPEED_ORDER:
    spd_ms = TM_SPEEDS_LABEL[spd]
    print(f"\n  Processing: Treadmill {spd} ({spd_ms} m/s)...")
    plot_population_gait(
        condition_label = f"Population — Treadmill {spd.capitalize()} "
                          f"({spd_ms} m/s)",
        loco       = "Treadmill",
        cond       = spd,
        side_key_R = "R",
        side_key_L = "L",
        save_path  = os.path.join(
            OUTPUT_DIR, f"fig_gait_treadmill_{spd}.png")
    )

# Overground straight — 4 speeds
print("="*60)
print("OVERGROUND — Population Gait Cycles (turn-filtered)")
print("="*60)
for spd in SPEED_ORDER:
    spd_ms = OG_SPEEDS_LABEL[spd]
    print(f"\n  Processing: Overground {spd} ({spd_ms} m/s)...")
    plot_population_gait(
        condition_label = f"Population — Overground {spd.capitalize()} "
                          f"({spd_ms} m/s, turn-filtered)",
        loco       = "Overground",
        cond       = spd,
        side_key_R = "R",
        side_key_L = "L",
        save_path  = os.path.join(
            OUTPUT_DIR, f"fig_gait_overground_{spd}.png")
    )

# Overground special
print("="*60)
print("OVERGROUND SPECIAL — Round & Obstacles")
print("="*60)
for cond, label in [("round","Round"), ("obstacles","Obstacles")]:
    print(f"\n  Processing: Overground {label}...")
    plot_population_gait(
        condition_label = f"Population — Overground {label}",
        loco       = "Overground",
        cond       = cond,
        side_key_R = "R",
        side_key_L = "L",
        save_path  = os.path.join(
            OUTPUT_DIR, f"fig_gait_overground_{cond}.png")
    )

# Slope
print("="*60)
print("SLOPE — Ascent & Descent")
print("="*60)
for rep in ["1", "2"]:
    for direction, sk_R, sk_L, label in [
        ("Ascent",  "asc_R",  "asc_L",  "ascent"),
        ("Descent", "desc_R", "desc_L", "descent"),
    ]:
        print(f"\n  Processing: Slope Rep{rep} {direction}...")
        plot_population_gait(
            condition_label = f"Population — Slope Rep{rep} {direction}",
            loco       = "Slope",
            cond       = rep,
            side_key_R = sk_R,
            side_key_L = sk_L,
            save_path  = os.path.join(
                OUTPUT_DIR,
                f"fig_gait_slope_rep{rep}_{label}.png")
        )

# Stair
print("="*60)
print("STAIR — Ascent & Descent")
print("="*60)
for rep in ["1", "2"]:
    for direction, sk_R, sk_L, label in [
        ("Ascent",  "asc_R",  "asc_L",  "ascent"),
        ("Descent", "desc_R", "desc_L", "descent"),
    ]:
        print(f"\n  Processing: Stair Rep{rep} {direction}...")
        plot_population_gait(
            condition_label = f"Population — Stair Rep{rep} {direction}",
            loco       = "Stair",
            cond       = rep,
            side_key_R = sk_R,
            side_key_L = sk_L,
            save_path  = os.path.join(
                OUTPUT_DIR,
                f"fig_gait_stair_rep{rep}_{label}.png")
        )

print("\n" + "="*60)
print("✅ All population gait cycle figures saved.")
figs = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("fig_gait")]
print(f"   Total figures: {len(figs)}")
for f in sorted(figs):
    print(f"   {f}")

# Cell 7 — Joint ROM Summary Statistics: Population mean ± SD per condition

In [ ]:
# ── Compute ROM for all participants, all conditions ──────────────────────────
print("="*60)
print("Computing ROM for all participants and conditions...")
print("="*60)

ALL_CONDITIONS_ROM = []
for spd in SPEED_ORDER:
    ALL_CONDITIONS_ROM.append(
        ("Treadmill",  spd, "R", "L", f"Treadmill {spd}"))
    ALL_CONDITIONS_ROM.append(
        ("Overground", spd, "R", "L", f"Overground {spd}"))
ALL_CONDITIONS_ROM.append(
    ("Overground", "round",     "R", "L", "Overground Round"))
ALL_CONDITIONS_ROM.append(
    ("Overground", "obstacles", "R", "L", "Overground Obstacles"))
for rep in ["1", "2"]:
    ALL_CONDITIONS_ROM.append(
        ("Slope", rep, "asc_R",  "asc_L",  f"Slope Rep{rep} Ascent"))
    ALL_CONDITIONS_ROM.append(
        ("Slope", rep, "desc_R", "desc_L", f"Slope Rep{rep} Descent"))
    ALL_CONDITIONS_ROM.append(
        ("Stair", rep, "asc_R",  "asc_L",  f"Stair Rep{rep} Ascent"))
    ALL_CONDITIONS_ROM.append(
        ("Stair", rep, "desc_R", "desc_L", f"Stair Rep{rep} Descent"))

summary_rows = []
for loco, cond, sk_R, sk_L, label in ALL_CONDITIONS_ROM:
    for jR, jname in STAT_JOINTS_R.items():
        jL     = R_TO_L[jR]
        roms_R, roms_L = [], []
        for plabel in P_LABELS:
            pdata = get_participant_rom(plabel, loco, cond, sk_R, sk_L)
            if pdata is None:
                continue
            if jR in pdata["rom_R"]:
                roms_R.append(pdata["rom_R"][jR])
            if jL in pdata["rom_L"]:
                roms_L.append(pdata["rom_L"][jL])
        for side_label, roms in [("R", roms_R), ("L", roms_L)]:
            n = len(roms)
            summary_rows.append({
                "Condition": label,
                "Joint":     jname,
                "Side":      side_label,
                "N":         n,
                "ROM_Mean":  round(np.mean(roms), 2)        if n     else np.nan,
                "ROM_SD":    round(np.std(roms, ddof=1), 2) if n > 1 else np.nan,
                "ROM_Min":   round(np.min(roms), 2)         if n     else np.nan,
                "ROM_Max":   round(np.max(roms), 2)         if n     else np.nan,
            })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "population_summary_stats.csv"), index=False)
print(f"✅ Saved: population_summary_stats.csv ({len(summary_df)} rows)")

# ── Paper-ready ROM table ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("PAPER ROM TABLE — Right Leg, Mean ± SD (°)")
print("="*60)

paper_cond_order = (
    [f"Treadmill {s}"  for s in SPEED_ORDER] +
    [f"Overground {s}" for s in SPEED_ORDER] +
    ["Overground Round", "Overground Obstacles"] +
    ["Slope Ascent", "Slope Descent",
     "Stair Ascent", "Stair Descent"]
)

paper_rows = []
for cond_label in paper_cond_order:
    for jname in JOINT_ORDER:
        if "Slope" in cond_label or "Stair" in cond_label:
            loco      = "Slope" if "Slope" in cond_label else "Stair"
            direction = "Ascent" if "Ascent" in cond_label else "Descent"
            r1 = summary_df[
                (summary_df.Side == "R") &
                (summary_df.Condition == f"{loco} Rep1 {direction}") &
                (summary_df.Joint == jname)]
            r2 = summary_df[
                (summary_df.Side == "R") &
                (summary_df.Condition == f"{loco} Rep2 {direction}") &
                (summary_df.Joint == jname)]
            if not r1.empty and not r2.empty:
                mean_avg = (r1.iloc[0].ROM_Mean + r2.iloc[0].ROM_Mean) / 2
                sd_avg   = np.sqrt((r1.iloc[0].ROM_SD**2 +
                                    r2.iloc[0].ROM_SD**2) / 2)
                paper_rows.append({
                    "Condition": cond_label,
                    "Joint":     jname,
                    "ROM_str":   f"{mean_avg:.1f} ± {sd_avg:.1f}",
                })
        else:
            sub = summary_df[
                (summary_df.Side == "R") &
                (summary_df.Condition == cond_label) &
                (summary_df.Joint == jname)]
            if not sub.empty:
                paper_rows.append({
                    "Condition": cond_label,
                    "Joint":     jname,
                    "ROM_str":   f"{sub.iloc[0].ROM_Mean:.1f} ± "
                                 f"{sub.iloc[0].ROM_SD:.1f}",
                })

paper_df = pd.DataFrame(paper_rows)
pivot    = paper_df.pivot(
    index="Condition", columns="Joint", values="ROM_str")
pivot    = pivot[JOINT_ORDER].reindex(paper_cond_order)
pivot.to_csv(os.path.join(OUTPUT_DIR, "paper_table_rom.csv"))

col_w = 20
print(f"\n  {'Condition':<25} ", end="")
for j in JOINT_ORDER:
    print(f"{j:>{col_w}}", end="")
print()
print("  " + "-"*(25 + col_w * len(JOINT_ORDER)))
for cond_label in paper_cond_order:
    if cond_label not in pivot.index:
        continue
    row = pivot.loc[cond_label]
    print(f"  {cond_label:<25} ", end="")
    for j in JOINT_ORDER:
        val = row[j] if not pd.isna(row[j]) else "N/A"
        print(f"{str(val):>{col_w}}", end="")
    print()
print(f"\n✅ Saved: paper_table_rom.csv")


# ── ROM Plots — 2 focused figures ────────────────────────────────────────────
from matplotlib.patches import Patch

right_df = summary_df[summary_df.Side == "R"].copy()

def get_rom_vals(cond_label, jname):
    """Helper to get mean and SD for a condition — handles averaged reps."""
    row = right_df[
        (right_df.Condition == cond_label) &
        (right_df.Joint == jname)]
    if not row.empty:
        return row.iloc[0].ROM_Mean, row.iloc[0].ROM_SD
    # Fall back to paper_df (averaged reps)
    prow = paper_df[
        (paper_df.Condition == cond_label) &
        (paper_df.Joint == jname)]
    if not prow.empty:
        parts = prow.iloc[0].ROM_str.split(" ± ")
        return float(parts[0]), float(parts[1])
    return np.nan, np.nan

def make_rom_barplot(ax, cond_labels, short_labels_map,
                     colors_list, jname, dividers=None,
                     ylabel=True):
    means, sds = [], []
    for cl in cond_labels:
        m, s = get_rom_vals(cl, jname)
        means.append(m if not np.isnan(m) else 0)
        sds.append(s   if not np.isnan(s) else 0)

    x = np.arange(len(cond_labels))
    ax.bar(x, means, yerr=sds,
           color=colors_list, capsize=4,
           edgecolor='black', linewidth=0.6,
           alpha=0.85,
           error_kw=dict(elinewidth=0.8, capthick=0.8))

    if dividers:
        for xpos in dividers:
            ax.axvline(xpos, color='gray', linestyle=':',
                       linewidth=0.8, alpha=0.6)

    xlabels = [short_labels_map.get(cl, cl) for cl in cond_labels]
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_title(jname, fontsize=11, fontweight='bold')
    if ylabel:
        ax.set_ylabel("ROM (°)", fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# ── Figure 1: Treadmill vs Overground (straight walking, speed effect) ────────
tm_og_conds = (
    [f"Treadmill {s}"  for s in SPEED_ORDER] +
    [f"Overground {s}" for s in SPEED_ORDER]
)
tm_og_short = {
    "Treadmill slow":    "TM\nSlow",
    "Treadmill normal":  "TM\nNorm",
    "Treadmill fast":    "TM\nFast",
    "Treadmill vfast":   "TM\nVFast",
    "Overground slow":   "OG\nSlow",
    "Overground normal": "OG\nNorm",
    "Overground fast":   "OG\nFast",
    "Overground vfast":  "OG\nVFast",
}
tm_og_colors = (
    [COLORS[0]] * 4 +   # Treadmill — blue
    [COLORS[1]] * 4     # Overground — orange
)
tm_og_legend = [
    Patch(facecolor=COLORS[0], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Treadmill"),
    Patch(facecolor=COLORS[1], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Overground (Straight)"),
]

fig1, axes1 = plt.subplots(1, 3, figsize=(14, 5))
fig1.suptitle(
    "ROM — Treadmill vs. Overground Straight Walking\n"
    "Right Leg, Mean ± SD (n=10)",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    make_rom_barplot(
        axes1[col_idx], tm_og_conds, tm_og_short,
        tm_og_colors, jname,
        dividers=[3.5],
        ylabel=(col_idx == 0))

fig1.legend(handles=tm_og_legend,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.04),
            ncol=2, fontsize=9,
            framealpha=0.9, edgecolor='#aaaaaa')
plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.subplots_adjust(wspace=0.25)
plt.savefig(os.path.join(OUTPUT_DIR, "fig_rom_treadmill_overground.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_rom_treadmill_overground.png")


# ── Figure 2: Special + Slope + Stair ─────────────────────────────────────────
special_conds = [
    "Overground Round", "Overground Obstacles",
    "Slope Ascent", "Slope Descent",
    "Stair Ascent", "Stair Descent",
]
special_short = {
    "Overground Round":     "OG\nRound",
    "Overground Obstacles": "OG\nObs",
    "Slope Ascent":         "Slope\nAsc",
    "Slope Descent":        "Slope\nDes",
    "Stair Ascent":         "Stair\nAsc",
    "Stair Descent":        "Stair\nDes",
}
special_colors = [
    COLORS[2],  # OG Round
    COLORS[3],  # OG Obstacles
    COLORS[4],  # Slope Asc
    COLORS[5],  # Slope Des
    COLORS[6],  # Stair Asc
    COLORS[7],  # Stair Des
]
special_legend = [
    Patch(facecolor=COLORS[2], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="OG Round"),
    Patch(facecolor=COLORS[3], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="OG Obstacles"),
    Patch(facecolor=COLORS[4], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Slope Ascent"),
    Patch(facecolor=COLORS[5], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Slope Descent"),
    Patch(facecolor=COLORS[6], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Stair Ascent"),
    Patch(facecolor=COLORS[7], edgecolor='black',
          linewidth=0.6, alpha=0.85, label="Stair Descent"),
]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5))
fig2.suptitle(
    "ROM — Overground Special, Slope & Stair Conditions\n"
    "Right Leg, Mean ± SD (n=10)",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    make_rom_barplot(
        axes2[col_idx], special_conds, special_short,
        special_colors, jname,
        dividers=[1.5, 3.5],
        ylabel=(col_idx == 0))

fig2.legend(handles=special_legend,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.04),
            ncol=6, fontsize=9,
            framealpha=0.9, edgecolor='#aaaaaa')
plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.subplots_adjust(wspace=0.25)
plt.savefig(os.path.join(OUTPUT_DIR, "fig_rom_special_slope_stair.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_rom_special_slope_stair.png")

# Cell 8 — Test-Retest Reliability: ICC(2,1) for Slope and Stair (Rep1 vs Rep2)

In [ ]:
# ── DEBUG: Check ICC data collection ─────────────────────────────────────────
print("Debugging ICC data collection...")

for loco in ["Slope", "Stair"]:
    for sk_R, sk_L, dir_label in [
        ("asc_R",  "asc_L",  "Ascent"),
        ("desc_R", "desc_L", "Descent"),
    ]:
        print(f"\n  {loco} {dir_label} | sk_R={sk_R}")
        for plabel in P_LABELS[:2]:  # check first 2 subjects only
            p1 = get_participant_rom(
                plabel, loco, "1", sk_R, sk_L)
            p2 = get_participant_rom(
                plabel, loco, "2", sk_R, sk_L)
            print(f"    {plabel}: Rep1={'✅' if p1 else '❌'} "
                  f"Rep2={'✅' if p2 else '❌'}")
            if p1:
                print(f"      rom_R keys: {list(p1['rom_R'].keys())[:3]}")

In [ ]:
# ── DEBUG: Check pingouin ICC output ─────────────────────────────────────────
import pingouin as pg

# Build a test dataframe with P01 slope ascent hip flexion
p1 = get_participant_rom("P01", "Slope", "1", "asc_R", "asc_L")
p2 = get_participant_rom("P01", "Slope", "2", "asc_R", "asc_L")

rep1_vals, rep2_vals = [], []
for plabel in P_LABELS:
    p1 = get_participant_rom(plabel, "Slope", "1", "asc_R", "asc_L")
    p2 = get_participant_rom(plabel, "Slope", "2", "asc_R", "asc_L")
    if p1 and p2:
        jcol = "hip_flexion_r"
        if jcol in p1["rom_R"] and jcol in p2["rom_R"]:
            rep1_vals.append(p1["rom_R"][jcol])
            rep2_vals.append(p2["rom_R"][jcol])

n = len(rep1_vals)
print(f"n={n}, rep1={rep1_vals[:3]}, rep2={rep2_vals[:3]}")

long_df = pd.DataFrame({
    "subject": list(range(n)) * 2,
    "rater":   ["rep1"] * n + ["rep2"] * n,
    "value":   rep1_vals + rep2_vals,
})

res = pg.intraclass_corr(
    data=long_df,
    targets="subject",
    raters="rater",
    ratings="value")

print("\nFull ICC output:")
print(res)
print("\nType column values:", res["Type"].tolist())

In [ ]:
# ── ICC(2,1) computation — Slope and Stair, Rep1 vs Rep2 ─────────────────────
print("="*60)
print("ICC(2,1) TEST-RETEST RELIABILITY")
print("Slope and Stair — Rep1 vs Rep2, per joint per side")
print("="*60)

icc_rows = []

for loco in ["Slope", "Stair"]:
    for direction, sk_R, sk_L, dir_label in [
        ("Ascent",  "asc_R",  "asc_L",  "Ascent"),
        ("Descent", "desc_R", "desc_L", "Descent"),
    ]:
        for jR, jname in STAT_JOINTS_R.items():
            jL = R_TO_L[jR]

            for side_label, jcol in [("R", jR), ("L", jL)]:

                rep1_vals, rep2_vals = [], []

                for plabel in P_LABELS:
                    p1 = get_participant_rom(
                        plabel, loco, "1", sk_R, sk_L)
                    p2 = get_participant_rom(
                        plabel, loco, "2", sk_R, sk_L)

                    if p1 is None or p2 is None:
                        continue

                    rom_key = f"rom_{side_label}"
                    if (jcol in p1.get(rom_key, {}) and
                            jcol in p2.get(rom_key, {})):
                        rep1_vals.append(p1[rom_key][jcol])
                        rep2_vals.append(p2[rom_key][jcol])

                n = len(rep1_vals)
                print(f"  {loco:5s} {dir_label:8s} | "
                      f"{jname:15s} ({side_label}) | "
                      f"n={n}", end="")

                if n >= 3:
                    long_df = pd.DataFrame({
                        "subject": list(range(n)) * 2,
                        "rater":   ["rep1"] * n + ["rep2"] * n,
                        "value":   rep1_vals + rep2_vals,
                    })
                    try:
                        res = pg.intraclass_corr(
                            data=long_df,
                            targets="subject",
                            raters="rater",
                            ratings="value")
                        # ── FIX: correct type string ──────────────
                        # pingouin returns "ICC(A,1)" for ICC(2,1)
                        icc = float(
                            res[res["Type"] == "ICC(A,1)"][
                                "ICC"].values[0])
                    except Exception as e:
                        print(f" | ICC error: {e}")
                        icc = np.nan

                    try:
                        _, t_pval = stats.ttest_rel(
                            rep1_vals, rep2_vals)
                    except:
                        t_pval = np.nan

                    if not np.isnan(icc):
                        icc_clipped = np.clip(icc, -1, 1)
                        sem = (np.std(rep1_vals + rep2_vals) *
                               np.sqrt(max(0, 1 - icc_clipped)))
                        mean_diff = np.mean(
                            np.array(rep1_vals) -
                            np.array(rep2_vals))
                    else:
                        sem = mean_diff = np.nan
                else:
                    icc = sem = t_pval = mean_diff = np.nan

                rel = ("Excellent" if not np.isnan(icc)
                                      and icc >= 0.90 else
                       "Good"      if not np.isnan(icc)
                                      and icc >= 0.75 else
                       "Moderate"  if not np.isnan(icc)
                                      and icc >= 0.50 else
                       "Poor"      if not np.isnan(icc)
                       else "N/A")

                icc_str = (f"{icc:.3f}"
                           if not np.isnan(icc) else "N/A")
                print(f" | ICC={icc_str:>6} | {rel}")

                icc_rows.append({
                    "Locomotion":    loco,
                    "Direction":     dir_label,
                    "Joint":         jname,
                    "Side":          side_label,
                    "N_pairs":       n,
                    "ROM_Rep1_Mean": round(np.mean(rep1_vals), 2)
                                     if n else np.nan,
                    "ROM_Rep1_SD":   round(np.std(rep1_vals,
                                                   ddof=1), 2)
                                     if n > 1 else np.nan,
                    "ROM_Rep2_Mean": round(np.mean(rep2_vals), 2)
                                     if n else np.nan,
                    "ROM_Rep2_SD":   round(np.std(rep2_vals,
                                                   ddof=1), 2)
                                     if n > 1 else np.nan,
                    "Mean_Diff_deg": round(mean_diff, 2)
                                     if not np.isnan(mean_diff)
                                     else np.nan,
                    "ICC_2_1":       round(icc, 3)
                                     if not np.isnan(icc)
                                     else np.nan,
                    "SEM_deg":       round(sem, 2)
                                     if not np.isnan(sem)
                                     else np.nan,
                    "Paired_t_p":    round(t_pval, 4)
                                     if not np.isnan(t_pval)
                                     else np.nan,
                    "Reliability":   rel,
                })

icc_df = pd.DataFrame(icc_rows)
icc_df.to_csv(
    os.path.join(OUTPUT_DIR,
                 "population_icc_testretest.csv"),
    index=False)
print(f"\n✅ Saved: population_icc_testretest.csv "
      f"({len(icc_df)} rows)")

# ── ICC Heatmap ───────────────────────────────────────────────────────────────
JOINT_SIDE_ORDER = [
    "Hip Flexion (R)", "Hip Flexion (L)",
    "Knee (R)",        "Knee (L)",
    "Ankle (R)",       "Ankle (L)",
]
ICC_COL_ORDER = [
    "Slope Ascent", "Slope Descent",
    "Stair Ascent", "Stair Descent",
]

icc_plot = icc_df.copy()
icc_plot["Label"]      = (icc_plot["Locomotion"] + " " +
                           icc_plot["Direction"])
icc_plot["Joint_Side"] = (icc_plot["Joint"] + " (" +
                           icc_plot["Side"] + ")")

heat = icc_plot.pivot(
    index="Joint_Side", columns="Label",
    values="ICC_2_1")
heat = heat.reindex(
    index=JOINT_SIDE_ORDER,
    columns=ICC_COL_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heat,
            annot=True, fmt=".2f",
            cmap="RdYlGn",
            vmin=0, vmax=1,
            linewidths=0.5,
            linecolor='white',
            annot_kws={"size": 10, "weight": "bold"},
            ax=ax,
            cbar_kws={"label": "ICC(2,1)",
                      "shrink": 0.8})

ax.set_title(
    "Test-Retest Reliability — ICC(2,1)\n"
    "Slope and Stair: Rep1 vs Rep2",
    fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
ax.set_yticklabels(
    ax.get_yticklabels(), fontsize=10, rotation=0)

for i, jside in enumerate(JOINT_SIDE_ORDER):
    row_icc = heat.loc[jside]
    for j, col in enumerate(ICC_COL_ORDER):
        val = row_icc[col]
        if not np.isnan(val):
            rel = ("E" if val >= 0.90 else
                   "G" if val >= 0.75 else
                   "M" if val >= 0.50 else "P")
            ax.text(j + 0.82, i + 0.82, rel,
                    ha='right', va='bottom',
                    fontsize=7, color='#333333',
                    style='italic')

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "fig_icc_heatmap.png"),
    dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_icc_heatmap.png")

# ── Reliability summary ───────────────────────────────────────────────────────
print("\n── Reliability Summary ──────────────────────────")
print(f"  {'Category':<12} {'Count':>6} {'%':>6}")
print("  " + "-"*28)
rel_counts = icc_df["Reliability"].value_counts()
total      = len(icc_df)
for cat in ["Excellent", "Good", "Moderate", "Poor", "N/A"]:
    cnt = rel_counts.get(cat, 0)
    print(f"  {cat:<12} {cnt:>6} "
          f"{cnt/total*100:>5.1f}%")
valid = icc_df.ICC_2_1.dropna()
if len(valid):
    print(f"\n  ICC range : "
          f"{valid.min():.3f} – {valid.max():.3f}")
    print(f"  ICC mean  : "
          f"{valid.mean():.3f} ± {valid.std():.3f}")

# Cell 9 — Speed Effect on ROM: Friedman test + Wilcoxon post-hoc (Bonferroni)

In [ ]:
# ── Friedman + Wilcoxon post-hoc ──────────────────────────────────────────────
print("="*60)
print("SPEED EFFECT ON ROM — Friedman + Wilcoxon/Bonferroni")
print("Treadmill and Overground, Right and Left leg")
print("="*60)

speed_rows = []

for loco, spd_ms_map in [
        ("Treadmill",  TM_SPEEDS_MS),
        ("Overground", OG_SPEEDS_MS)]:

    for jR, jname in STAT_JOINTS_R.items():
        jL = R_TO_L[jR]

        for side_label, jcol in [("R", jR), ("L", jL)]:

            # Build paired matrix: subjects × speeds
            per_subject = {plabel: {} for plabel in P_LABELS}
            for spd in SPEED_ORDER:
                for plabel in P_LABELS:
                    pdata = get_participant_rom(
                        plabel, loco, spd, "R", "L")
                    if pdata and jcol in pdata[f"rom_{side_label}"]:
                        per_subject[plabel][spd] = \
                            pdata[f"rom_{side_label}"][jcol]

            # Keep only subjects with data at ALL 4 speeds
            complete = [p for p in P_LABELS
                        if all(s in per_subject[p]
                               for s in SPEED_ORDER)]
            n = len(complete)

            if n < 3:
                print(f"  {loco:12s} | {jname:15s} ({side_label}) | "
                      f"Insufficient data (n={n})")
                continue

            groups     = [[per_subject[p][s] for p in complete]
                          for s in SPEED_ORDER]
            means_sds  = [(round(np.mean(g), 2),
                           round(np.std(g, ddof=1), 2))
                          for g in groups]

            # Friedman test
            try:
                chi2, pval = friedmanchisquare(*groups)
            except:
                chi2, pval = np.nan, np.nan

            k = len(SPEED_ORDER)
            W = (chi2 / (n * (k - 1))
                 if not np.isnan(chi2) else np.nan)

            sig = "*" if not np.isnan(pval) and pval < 0.05 else "ns"
            print(f"\n  {loco:12s} | {jname:15s} ({side_label}) | "
                  f"χ²={chi2:.3f}, p={pval:.4f} {sig}, "
                  f"W={W:.3f} (n={n})")

            # Wilcoxon post-hoc with Bonferroni correction
            speed_pairs = list(combinations(range(k), 2))
            n_pairs     = len(speed_pairs)
            posthoc     = []

            for i, j in speed_pairs:
                g1, g2 = groups[i], groups[j]
                try:
                    stat_w, p_w = wilcoxon(
                        g1, g2, zero_method='wilcox')
                    p_bonf = min(p_w * n_pairs, 1.0)
                except:
                    stat_w, p_w, p_bonf = np.nan, np.nan, np.nan

                sig_pair = ("*" if not np.isnan(p_bonf)
                            and p_bonf < 0.05 else "ns")
                delta    = (means_sds[j][0] - means_sds[i][0])
                print(f"    {SPEED_ORDER[i]:>6} vs {SPEED_ORDER[j]:<6} | "
                      f"Δ={delta:+.1f}° | "
                      f"p_bonf={p_bonf:.4f} {sig_pair}")
                posthoc.append({
                    "spd_A":  SPEED_ORDER[i],
                    "spd_B":  SPEED_ORDER[j],
                    "mean_A": means_sds[i][0],
                    "mean_B": means_sds[j][0],
                    "p_bonf": round(p_bonf, 4)
                              if not np.isnan(p_bonf) else np.nan,
                    "sig":    sig_pair,
                })

            row = {
                "Locomotion":    loco,
                "Joint":         jname,
                "Side":          side_label,
                "N_complete":    n,
                "Friedman_chi2": round(chi2, 3) if not np.isnan(chi2) else np.nan,
                "Friedman_p":    round(pval, 4)  if not np.isnan(pval) else np.nan,
                "Kendall_W":     round(W, 3)     if not np.isnan(W)    else np.nan,
                "Significant":   "Yes" if not np.isnan(pval)
                                 and pval < 0.05 else "No",
            }
            for i, spd in enumerate(SPEED_ORDER):
                row[f"ROM_Mean_{spd}"] = means_sds[i][0]
                row[f"ROM_SD_{spd}"]   = means_sds[i][1]
                row[f"Speed_{spd}_ms"] = spd_ms_map[spd]
            for ph in posthoc:
                col = f"p_bonf_{ph['spd_A']}_vs_{ph['spd_B']}"
                row[col]              = ph["p_bonf"]
                row[col.replace("p_bonf","sig")] = ph["sig"]

            speed_rows.append(row)

speed_effect_df = pd.DataFrame(speed_rows)
speed_effect_df.to_csv(
    os.path.join(OUTPUT_DIR, "population_speed_effect_friedman.csv"),
    index=False)
print(f"\n✅ Saved: population_speed_effect_friedman.csv")

# ── Friedman summary table ────────────────────────────────────────────────────
print("\n── Friedman Summary ──────────────────────────────────────────")
print(f"  {'Locomotion':<12} {'Joint':<15} {'Side':>4} | "
      f"{'χ²':>7} {'p':>8} {'W':>6} {'Sig':>5}")
print("  " + "-"*60)
for _, row in speed_effect_df.iterrows():
    chi2_s = f"{row.Friedman_chi2:.3f}" if not pd.isna(row.Friedman_chi2) else "N/A"
    p_s    = f"{row.Friedman_p:.4f}"    if not pd.isna(row.Friedman_p)    else "N/A"
    W_s    = f"{row.Kendall_W:.3f}"     if not pd.isna(row.Kendall_W)     else "N/A"
    sig_s  = "*" if row.Significant == "Yes" else "ns"
    print(f"  {row.Locomotion:<12} {row.Joint:<15} {row.Side:>4} | "
          f"{chi2_s:>7} {p_s:>8} {W_s:>6} {sig_s:>5}")


# ── Speed effect bar plots ────────────────────────────────────────────────────
def add_significance_bracket(ax, x1, x2, y, p_bonf, h=0.5, fontsize=8):
    if np.isnan(p_bonf) or p_bonf >= 0.05:
        return
    sig_text = ("***" if p_bonf < 0.001 else
                "**"  if p_bonf < 0.01  else "*")
    ax.plot([x1, x1, x2, x2],
            [y, y+h, y+h, y],
            lw=0.9, color='black')
    ax.text((x1+x2)/2, y+h+0.1, sig_text,
            ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold')


for loco, spd_map in [("Treadmill",  TM_SPEEDS_LABEL),
                       ("Overground", OG_SPEEDS_LABEL)]:

    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    fig.suptitle(
        f"Speed Effect on ROM — {loco}\n"
        f"Friedman test | Post-hoc: Wilcoxon/Bonferroni (n=10)",
        fontsize=12, fontweight='bold')

    for row_idx, (side_label, side_name) in enumerate(
            [("R","Right Leg"), ("L","Left Leg")]):

        loco_df = speed_effect_df[
            (speed_effect_df.Locomotion == loco) &
            (speed_effect_df.Side == side_label)]

        for col_idx, jname in enumerate(JOINT_ORDER):
            ax  = axes[row_idx, col_idx]
            row = loco_df[loco_df.Joint == jname]
            if row.empty:
                continue
            row = row.iloc[0]

            means_ = [row[f"ROM_Mean_{s}"] for s in SPEED_ORDER]
            sds_   = [row[f"ROM_SD_{s}"]   for s in SPEED_ORDER]
            xlbls  = [f"{s.capitalize()}\n({spd_map[s]} m/s)"
                      for s in SPEED_ORDER]
            x      = np.arange(4)

            bars = ax.bar(x, means_, yerr=sds_,
                          color=COLORS[:4],
                          capsize=4, edgecolor='black',
                          linewidth=0.6, alpha=0.85,
                          error_kw=dict(elinewidth=0.8))

            # Value labels on bars
            for xi, (m, s) in enumerate(zip(means_, sds_)):
                if not np.isnan(m):
                    ax.text(xi, m + s + 0.3,
                            f"{m:.1f}°",
                            ha='center', va='bottom',
                            fontsize=7.5, fontweight='bold')

            ax.set_xticks(x)
            ax.set_xticklabels(xlbls, fontsize=8)

            if row_idx == 0:
                ax.set_title(jname, fontsize=11, fontweight='bold')
            if col_idx == 0:
                ax.set_ylabel(f"{side_name}\nROM (°)", fontsize=9)

            # Friedman p in subtitle
            if not pd.isna(row.Friedman_p):
                sig_str = "*" if row.Significant == "Yes" else "ns"
                ax.set_xlabel(
                    f"χ²({3})={row.Friedman_chi2:.2f}, "
                    f"p={row.Friedman_p:.4f} {sig_str}, "
                    f"W={row.Kendall_W:.2f}",
                    fontsize=7.5, color='navy')

            ax.grid(True, alpha=0.3, axis='y')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.set_axisbelow(True)

            # Significance brackets (only if Friedman significant)
            if row.Significant != "Yes":
                continue

            sig_pairs = []
            for i, j in combinations(range(4), 2):
                spd_A = SPEED_ORDER[i]
                spd_B = SPEED_ORDER[j]
                col   = f"p_bonf_{spd_A}_vs_{spd_B}"
                if col in row.index and not pd.isna(row[col]):
                    if row[col] < 0.05:
                        sig_pairs.append((i, j, row[col]))

            if not sig_pairs:
                continue

            max_top = max(
                m + s for m, s in zip(means_, sds_)
                if not np.isnan(m) and not np.isnan(s))
            base  = max_top + 2.5
            step  = 2.0
            pairs_sorted = sorted(
                sig_pairs, key=lambda t: abs(t[1]-t[0]))

            for rank, (i, j, p_b) in enumerate(pairs_sorted):
                add_significance_bracket(
                    ax, i, j, base + rank * step,
                    p_b, h=0.6, fontsize=8)

            new_top = base + len(pairs_sorted) * step + 3.0
            if new_top > ax.get_ylim()[1]:
                ax.set_ylim(top=new_top)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.subplots_adjust(hspace=0.35, wspace=0.30)
    fname = f"fig_speed_effect_{loco.lower()}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {fname}")

In [ ]:
# ── Speed effect bar plots — 2×6 compact layout ───────────────────────────────
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EASY CONTROLS — adjust these two values only
FONT_SCALE   = 1.7  # increase to make all fonts bigger (1.0=small, 1.5=medium, 2.0=large)
TIGHT_SCALE  = 0.80  # decrease to tighten spacing (1.0=normal, 0.7=tight, 0.5=very tight)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── CANONICAL SPEEDS (matching Supplementary Tables S1/S2 and main manuscript) ─
# These override TM_SPEEDS_LABEL / OG_SPEEDS_LABEL for figure display only,
# to keep Figure 8 consistent with reported per-subject speeds in S1.
TM_SPEEDS_DISPLAY = {
    "slow":   "0.73",
    "normal": "1.11",
    "fast":   "1.49",
    "vfast":  "1.87",
}
OG_SPEEDS_DISPLAY = {
    "slow":   "0.67",
    "normal": "0.79",
    "fast":   "0.99",
    "vfast":  "1.16",
}

# Derived font sizes
F_title   = 10  * FONT_SCALE
F_ylabel  = 9   * FONT_SCALE
F_xlabel  = 8   * FONT_SCALE
F_xtick   = 8   * FONT_SCALE
F_ytick   = 8   * FONT_SCALE
F_bar_lbl = 7.5 * FONT_SCALE
F_bracket = 8   * FONT_SCALE

# Derived spacing
H_SPACE  = 0.35 * TIGHT_SCALE
W_SPACE  = 0.22 * TIGHT_SCALE


def add_significance_bracket(ax, x1, x2, y, p_bonf,
                              h=0.4, fontsize=8):
    if np.isnan(p_bonf) or p_bonf >= 0.05:
        return
    sig_text = ("***" if p_bonf < 0.001 else
                "**"  if p_bonf < 0.01  else "*")
    ax.plot([x1, x1, x2, x2],
            [y, y+h, y+h, y],
            lw=0.8, color='black', clip_on=False)
    ax.text((x1+x2)/2, y+h+0.05, sig_text,
            ha='center', va='bottom',
            fontsize=fontsize, fontweight='bold',
            clip_on=False)


fig, axes = plt.subplots(2, 6, figsize=(22, 10),
                          sharex=False, sharey=False)

row_configs = [
    ("Treadmill",  TM_SPEEDS_DISPLAY, "Treadmill"),
    ("Overground", OG_SPEEDS_DISPLAY, "Overground"),
]

col_configs = [
    ("Hip Flexion", "R", "hip_flexion_r"),
    ("Knee",        "R", "knee_angle_r"),
    ("Ankle",       "R", "ankle_angle_r"),
    ("Hip Flexion", "L", "hip_flexion_l"),
    ("Knee",        "L", "knee_angle_l"),
    ("Ankle",       "L", "ankle_angle_l"),
]

spd_short = {
    "slow":   "S",
    "normal": "N",
    "fast":   "F",
    "vfast":  "VF",
}

for row_idx, (loco, spd_map, row_label) in \
        enumerate(row_configs):

    loco_df = speed_effect_df[
        speed_effect_df.Locomotion == loco]

    for col_idx, (jname, side, jcol) in \
            enumerate(col_configs):

        ax  = axes[row_idx, col_idx]
        sub = loco_df[
            (loco_df.Joint == jname) &
            (loco_df.Side  == side)]

        if sub.empty:
            ax.axis('off')
            continue
        row = sub.iloc[0]

        means_ = [row[f"ROM_Mean_{s}"] for s in SPEED_ORDER]
        sds_   = [row[f"ROM_SD_{s}"]   for s in SPEED_ORDER]
        xlbls  = [f"{spd_short[s]}\n({spd_map[s]})"
                  for s in SPEED_ORDER]
        x      = np.arange(4)

        # ── Bars ──────────────────────────────────────────────
        ax.bar(x, means_, yerr=sds_,
               color=COLORS[:4],
               capsize=3,
               edgecolor='black',
               linewidth=0.6,
               alpha=0.85,
               width=0.55,
               error_kw=dict(elinewidth=0.8,
                             capthick=0.8,
                             zorder=5))

        # Value labels
        for xi, (m, s) in enumerate(zip(means_, sds_)):
            if not np.isnan(m):
                ax.text(xi, m + s + 0.2,
                        f"{m:.0f}°",
                        ha='center', va='bottom',
                        fontsize=F_bar_lbl,
                        fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels(xlbls, fontsize=F_xtick)
        ax.set_xlim(-0.55, 3.55)

        # ── Y limits ──────────────────────────────────────────
        valid_means = [m for m in means_ if not np.isnan(m)]
        valid_sds   = [s for s in sds_   if not np.isnan(s)]
        y_min = 0
        y_max = max(valid_means) + max(valid_sds) * 5.0
        ax.set_ylim(y_min, y_max)

        # Column title (top row only)
        if row_idx == 0:
            side_str = "Right" if side == "R" else "Left"
            ax.set_title(f"{jname} ({side_str})",
                         fontsize=F_title,
                         fontweight='bold', pad=5)

        # Row label (first column only)
        if col_idx == 0:
            ax.set_ylabel(f"{row_label}\nROM (°)",
                          fontsize=F_ylabel)

        # Friedman stats as xlabel
        if not pd.isna(row.Friedman_p):
            sig_str = "*" if row.Significant == "Yes" \
                      else "ns"
            ax.set_xlabel(
                f"$p$={row.Friedman_p:.4f} {sig_str}, "
                f"$W$={row.Kendall_W:.2f}",
                fontsize=F_xlabel,
                color='navy', labelpad=3)

        ax.tick_params(axis='y', labelsize=F_ytick)
        ax.grid(True, alpha=0.2, axis='y',
                linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_axisbelow(True)

        # ── Significance brackets ──────────────────────────────
        if row.Significant != "Yes":
            continue

        sig_pairs = []
        for i, j in combinations(range(4), 2):
            spd_A   = SPEED_ORDER[i]
            spd_B   = SPEED_ORDER[j]
            col_key = f"p_bonf_{spd_A}_vs_{spd_B}"
            if (col_key in row.index and
                    not pd.isna(row[col_key]) and
                    row[col_key] < 0.05):
                sig_pairs.append((i, j, row[col_key]))

        if not sig_pairs:
            continue

        max_top = max(
            m + s for m, s in zip(means_, sds_)
            if not np.isnan(m) and not np.isnan(s))

        y_range = y_max - y_min
        base    = max_top + y_range * 0.04
        step    = y_range * 0.07

        sig_pairs_sorted = sorted(
            sig_pairs, key=lambda t: abs(t[1]-t[0]))

        for rank, (i, j, p_b) in \
                enumerate(sig_pairs_sorted):
            add_significance_bracket(
                ax, i, j,
                base + rank * step,
                p_b,
                h=y_range * 0.025,
                fontsize=F_bracket)

        new_top = base + \
                  len(sig_pairs_sorted) * step + \
                  y_range * 0.06
        if new_top > ax.get_ylim()[1]:
            ax.set_ylim(top=new_top)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.subplots_adjust(hspace=H_SPACE, wspace=W_SPACE)

fname = "fig_speed_effect_combined.png"
plt.savefig(os.path.join(OUTPUT_DIR, fname),
            dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {fname}")
print(f"   FONT_SCALE={FONT_SCALE} | TIGHT_SCALE={TIGHT_SCALE}")

# Cell 10 — Bilateral Symmetry: Symmetry Index (%) across all conditions

In [ ]:
# ── Bilateral Symmetry Index computation ──────────────────────────────────────
print("="*60)
print("BILATERAL SYMMETRY ANALYSIS")
print("Symmetry Index (%) — per joint, per condition")
print("="*60)

sym_rows = []

for loco, cond, sk_R, sk_L, label in ALL_CONDITIONS_ROM:
    for jR, jname in STAT_JOINTS_R.items():
        jL     = R_TO_L[jR]
        si_vals = []

        for plabel in P_LABELS:
            pdata = get_participant_rom(plabel, loco, cond, sk_R, sk_L)
            if pdata is None:
                continue
            if jR in pdata["rom_R"] and jL in pdata["rom_L"]:
                si = symmetry_index(
                    pdata["rom_L"][jL],
                    pdata["rom_R"][jR])
                if not np.isnan(si):
                    si_vals.append(si)

        n = len(si_vals)
        sym_rows.append({
            "Condition":   label,
            "Joint":       jname,
            "N":           n,
            "SI_Mean_pct": round(np.mean(si_vals),   2) if n     else np.nan,
            "SI_SD_pct":   round(np.std(si_vals, ddof=1), 2) if n > 1 else np.nan,
            "SI_Median":   round(np.median(si_vals), 2) if n     else np.nan,
            "SI_Max":      round(np.max(si_vals),    2) if n     else np.nan,
            "Symmetric":   "Yes" if n and np.mean(si_vals) < 10 else "No",
        })

sym_df = pd.DataFrame(sym_rows)
sym_df.to_csv(
    os.path.join(OUTPUT_DIR, "population_symmetry.csv"), index=False)
print(f"✅ Saved: population_symmetry.csv ({len(sym_df)} rows)")

# ── Print symmetry table ──────────────────────────────────────────────────────
print(f"\n  {'Condition':<25} {'Joint':<15} "
      f"{'SI Mean%':>9} {'SI SD%':>8} {'<10%?':>7}")
print("  " + "-"*68)
for _, row in sym_df.iterrows():
    if pd.isna(row.SI_Mean_pct):
        continue
    sym_str = "✅ Yes" if row.Symmetric == "Yes" else "⚠️  No"
    print(f"  {row.Condition:<25} {row.Joint:<15} "
          f"{row.SI_Mean_pct:>9.2f} {row.SI_SD_pct:>8.2f} "
          f"{sym_str:>7}")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n── Overall Symmetry Summary ─────────────────────────────────")
sym_yes = (sym_df.Symmetric == "Yes").sum()
sym_no  = (sym_df.Symmetric == "No").sum()
total   = sym_yes + sym_no
print(f"  Symmetric (SI < 10%) : {sym_yes}/{total} "
      f"({sym_yes/total*100:.1f}%)")
print(f"  SI range             : "
      f"{sym_df.SI_Mean_pct.min():.2f}% – "
      f"{sym_df.SI_Mean_pct.max():.2f}%")
print(f"  SI overall mean      : "
      f"{sym_df.SI_Mean_pct.mean():.2f}% ± "
      f"{sym_df.SI_Mean_pct.std():.2f}%")


# ── Figure 1: Treadmill + Overground symmetry ─────────────────────────────────
def get_mode_sym(cond_label):
    if "Treadmill"  in cond_label: return "Treadmill"
    if "Round"      in cond_label: return "OG Round"
    if "Obstacles"  in cond_label: return "OG Obstacles"
    if "Overground" in cond_label: return "Overground"
    if "Slope Ascent"  in cond_label: return "Slope Ascent"
    if "Slope Descent" in cond_label: return "Slope Descent"
    if "Stair Ascent"  in cond_label: return "Stair Ascent"
    if "Stair Descent" in cond_label: return "Stair Descent"
    return "Other"

sym_colors = {
    "Treadmill":    COLORS[0],
    "Overground":   COLORS[1],
    "OG Round":     COLORS[2],
    "OG Obstacles": COLORS[3],
    "Slope Ascent": COLORS[4],
    "Slope Descent":COLORS[5],
    "Stair Ascent": COLORS[6],
    "Stair Descent":COLORS[7],
}

# ── Figure 1: Treadmill + Overground straight ─────────────────────────────────
tm_og_sym_conds = (
    [f"Treadmill {s}"  for s in SPEED_ORDER] +
    [f"Overground {s}" for s in SPEED_ORDER]
)
tm_og_sym_short = {
    "Treadmill slow":    "TM\nSlow",
    "Treadmill normal":  "TM\nNorm",
    "Treadmill fast":    "TM\nFast",
    "Treadmill vfast":   "TM\nVFast",
    "Overground slow":   "OG\nSlow",
    "Overground normal": "OG\nNorm",
    "Overground fast":   "OG\nFast",
    "Overground vfast":  "OG\nVFast",
}
tm_og_sym_colors = [COLORS[0]]*4 + [COLORS[1]]*4

fig1, axes1 = plt.subplots(1, 3, figsize=(14, 5))
fig1.suptitle(
    "Bilateral Symmetry — Treadmill vs Overground\n"
    "Symmetry Index % (Mean ± SD, n=10). "
    "Dashed line = 10% threshold.",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    ax  = axes1[col_idx]
    sub = sym_df[sym_df.Joint == jname]

    means, sds, colors, xlabels = [], [], [], []
    for cl in tm_og_sym_conds:
        row = sub[sub.Condition == cl]
        means.append(row.iloc[0].SI_Mean_pct if not row.empty else 0)
        sds.append(row.iloc[0].SI_SD_pct     if not row.empty else 0)
        xlabels.append(tm_og_sym_short.get(cl, cl))
        colors.append(COLORS[0] if "Treadmill" in cl else COLORS[1])

    x = np.arange(len(means))
    ax.bar(x, means, yerr=sds,
           color=colors, capsize=4,
           edgecolor='black', linewidth=0.6,
           alpha=0.85,
           error_kw=dict(elinewidth=0.8))

    ax.axhline(10, color='red', linestyle='--',
               linewidth=1.2, alpha=0.8,
               label='10% threshold')
    ax.axvline(3.5, color='gray', linestyle=':',
               linewidth=0.8, alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_title(jname, fontsize=11, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel("Symmetry Index (%)", fontsize=10)
        ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
leg1 = [
    Patch(facecolor=COLORS[0], edgecolor='black',
          alpha=0.85, label="Treadmill"),
    Patch(facecolor=COLORS[1], edgecolor='black',
          alpha=0.85, label="Overground"),
]
fig1.legend(handles=leg1,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.04),
            ncol=2, fontsize=9,
            framealpha=0.9, edgecolor='#aaaaaa')
plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.subplots_adjust(wspace=0.25)
plt.savefig(os.path.join(OUTPUT_DIR, "fig_symmetry_tm_og.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_symmetry_tm_og.png")


# ── Figure 2: Special + Slope + Stair symmetry ────────────────────────────────
special_sym_conds = [
    "Overground Round", "Overground Obstacles",
    "Slope Ascent",     "Slope Descent",
    "Stair Ascent",     "Stair Descent",
]
special_sym_short = {
    "Overground Round":     "OG\nRound",
    "Overground Obstacles": "OG\nObs",
    "Slope Ascent":         "Slope\nAsc",
    "Slope Descent":        "Slope\nDes",
    "Stair Ascent":         "Stair\nAsc",
    "Stair Descent":        "Stair\nDes",
}
special_sym_colors = [
    COLORS[2], COLORS[3],
    COLORS[4], COLORS[5],
    COLORS[6], COLORS[7],
]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5))
fig2.suptitle(
    "Bilateral Symmetry — Special, Slope & Stair Conditions\n"
    "Symmetry Index % (Mean ± SD, n=10). "
    "Dashed line = 10% threshold.",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    ax  = axes2[col_idx]
    sub = sym_df[sym_df.Joint == jname]

    means, sds, xlabels = [], [], []
    for cl in special_sym_conds:
        row = sub[sub.Condition == cl]
        means.append(row.iloc[0].SI_Mean_pct if not row.empty else 0)
        sds.append(row.iloc[0].SI_SD_pct     if not row.empty else 0)
        xlabels.append(special_sym_short.get(cl, cl))

    x = np.arange(len(means))
    ax.bar(x, means, yerr=sds,
           color=special_sym_colors,
           capsize=4,
           edgecolor='black', linewidth=0.6,
           alpha=0.85,
           error_kw=dict(elinewidth=0.8))

    ax.axhline(10, color='red', linestyle='--',
               linewidth=1.2, alpha=0.8)
    ax.axvline(1.5, color='gray', linestyle=':',
               linewidth=0.8, alpha=0.6)
    ax.axvline(3.5, color='gray', linestyle=':',
               linewidth=0.8, alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=9)
    ax.set_title(jname, fontsize=11, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel("Symmetry Index (%)", fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

leg2 = [
    Patch(facecolor=COLORS[2], edgecolor='black',
          alpha=0.85, label="OG Round"),
    Patch(facecolor=COLORS[3], edgecolor='black',
          alpha=0.85, label="OG Obstacles"),
    Patch(facecolor=COLORS[4], edgecolor='black',
          alpha=0.85, label="Slope Ascent"),
    Patch(facecolor=COLORS[5], edgecolor='black',
          alpha=0.85, label="Slope Descent"),
    Patch(facecolor=COLORS[6], edgecolor='black',
          alpha=0.85, label="Stair Ascent"),
    Patch(facecolor=COLORS[7], edgecolor='black',
          alpha=0.85, label="Stair Descent"),
]
fig2.legend(handles=leg2,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.04),
            ncol=6, fontsize=9,
            framealpha=0.9, edgecolor='#aaaaaa')
plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.subplots_adjust(wspace=0.25)
plt.savefig(os.path.join(OUTPUT_DIR, "fig_symmetry_special_slope_stair.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_symmetry_special_slope_stair.png")

# Cell 11 — Condition Comparison: ROM across locomotion modes (Kruskal-Wallis + post-hoc)

In [ ]:
# ── Build mode-level ROM (averaged across speeds/reps per subject) ─────────────
print("="*60)
print("CONDITION COMPARISON — ROM across locomotion modes")
print("Kruskal-Wallis + Mann-Whitney post-hoc (Bonferroni)")
print("="*60)

# For each mode, average ROM across all conditions within that mode
mode_conditions_map = {
    "Treadmill":     [("Treadmill",  s, "R", "L")
                      for s in SPEED_ORDER],
    "Overground":    [("Overground", s, "R", "L")
                      for s in SPEED_ORDER],
    "OG Round":      [("Overground", "round",     "R", "L")],
    "OG Obstacles":  [("Overground", "obstacles", "R", "L")],
    "Slope Ascent":  [("Slope", r, "asc_R",  "asc_L")
                      for r in ["1","2"]],
    "Slope Descent": [("Slope", r, "desc_R", "desc_L")
                      for r in ["1","2"]],
    "Stair Ascent":  [("Stair", r, "asc_R",  "asc_L")
                      for r in ["1","2"]],
    "Stair Descent": [("Stair", r, "desc_R", "desc_L")
                      for r in ["1","2"]],
}

MODE_ORDER = list(mode_conditions_map.keys())

# Collect per-subject mean ROM per mode
mode_rom = {
    mode: {jname: [] for jname in JOINT_ORDER}
    for mode in MODE_ORDER
}

for mode, cond_list in mode_conditions_map.items():
    for plabel in P_LABELS:
        subj_vals = {jname: [] for jname in JOINT_ORDER}

        for loco, cond, sk_R, sk_L in cond_list:
            pdata = get_participant_rom(
                plabel, loco, cond, sk_R, sk_L)
            if pdata is None:
                continue
            for jR, jname in STAT_JOINTS_R.items():
                if jR in pdata["rom_R"]:
                    subj_vals[jname].append(pdata["rom_R"][jR])

        for jname in JOINT_ORDER:
            if subj_vals[jname]:
                mode_rom[mode][jname].append(
                    np.mean(subj_vals[jname]))

print("✅ Mode ROM collected.")
for mode in MODE_ORDER:
    n = len(mode_rom[mode][JOINT_ORDER[0]])
    print(f"  {mode:<16} n={n} subjects")

# ── Kruskal-Wallis test ───────────────────────────────────────────────────────
print(f"\n── Kruskal-Wallis Results ────────────────────────────────────")
print(f"  {'Joint':<15} {'H':>8} {'p':>9} {'Sig':>5}")
print("  " + "-"*42)

kw_results = {}
for jname in JOINT_ORDER:
    groups = [mode_rom[m][jname] for m in MODE_ORDER]
    valid  = [g for g in groups if len(g) >= 2]
    if len(valid) >= 2:
        H, pval = stats.kruskal(*valid)
        kw_results[jname] = {"H": H, "p": pval}
        sig = "*" if pval < 0.05 else "ns"
        print(f"  {jname:<15} {H:>8.3f} {pval:>9.4f} {sig:>5}")

# ── Mann-Whitney post-hoc (Bonferroni) ───────────────────────────────────────
print(f"\n── Post-hoc: Mann-Whitney U (Bonferroni corrected) ──────────")

cond_comp_rows = []
mode_pairs     = list(combinations(MODE_ORDER, 2))
n_pairs        = len(mode_pairs)

for jname in JOINT_ORDER:
    print(f"\n  {jname}:")
    for mA, mB in mode_pairs:
        g1 = mode_rom[mA][jname]
        g2 = mode_rom[mB][jname]
        if len(g1) < 2 or len(g2) < 2:
            continue
        _, p_raw  = stats.mannwhitneyu(
            g1, g2, alternative='two-sided')
        p_bonf    = min(p_raw * n_pairs, 1.0)
        sig       = "*" if p_bonf < 0.05 else "ns"
        mean_diff = np.mean(g1) - np.mean(g2)

        cond_comp_rows.append({
            "Joint":      jname,
            "Mode_A":     mA,
            "Mode_B":     mB,
            "Mean_A":     round(np.mean(g1), 2),
            "Mean_B":     round(np.mean(g2), 2),
            "Mean_Diff":  round(mean_diff, 2),
            "p_raw":      round(p_raw,  4),
            "p_bonf":     round(p_bonf, 4),
            "Significant":"Yes" if p_bonf < 0.05 else "No",
        })
        if p_bonf < 0.05:
            print(f"    {mA:<16} vs {mB:<16} | "
                  f"Δ={mean_diff:+.1f}° | "
                  f"p_bonf={p_bonf:.4f} {sig}")

cond_comp_df = pd.DataFrame(cond_comp_rows)
cond_comp_df.to_csv(
    os.path.join(OUTPUT_DIR, "condition_comparison_posthoc.csv"),
    index=False)
print(f"\n✅ Saved: condition_comparison_posthoc.csv")

# ── Figure 1: Boxplot — Treadmill + Overground ────────────────────────────────
tm_og_modes   = ["Treadmill", "Overground",
                  "OG Round", "OG Obstacles"]
tm_og_colors  = [COLORS[0], COLORS[1], COLORS[2], COLORS[3]]

fig1, axes1 = plt.subplots(1, 3, figsize=(14, 5))
fig1.suptitle(
    "ROM Comparison — Treadmill & Overground Conditions\n"
    "Right Leg (n=10), Individual subjects shown",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    ax        = axes1[col_idx]
    data_plot = [mode_rom[m][jname] for m in tm_og_modes]

    bp = ax.boxplot(data_plot,
                    patch_artist=True,
                    widths=0.5,
                    medianprops=dict(color='black',
                                     linewidth=2.0))
    for patch, color in zip(bp['boxes'], tm_og_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for element in ['whiskers','caps','fliers']:
        for item in bp[element]:
            item.set_color('#444444')
            item.set_linewidth(0.8)

    # Individual subject dots
    for xi, vals in enumerate(data_plot, start=1):
        jitter = np.random.uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jitter, vals,
                   color='black', s=18,
                   zorder=5, alpha=0.7)

    # KW result
    if jname in kw_results:
        H   = kw_results[jname]["H"]
        p   = kw_results[jname]["p"]
        sig = "*" if p < 0.05 else "ns"
        ax.set_xlabel(
            f"KW: H={H:.2f}, p={p:.4f} {sig}",
            fontsize=8, color='navy')

    ax.set_xticks(range(1, len(tm_og_modes)+1))
    ax.set_xticklabels(
        ["Treadmill", "Overground", "OG\nRound", "OG\nObs"],
        fontsize=9)
    ax.set_title(jname, fontsize=11, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel("ROM (°)", fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.subplots_adjust(wspace=0.28)
plt.savefig(
    os.path.join(OUTPUT_DIR, "fig_condition_comparison_tm_og.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_condition_comparison_tm_og.png")

# ── Figure 2: Boxplot — Slope + Stair ─────────────────────────────────────────
terrain_modes  = ["Slope Ascent", "Slope Descent",
                   "Stair Ascent", "Stair Descent"]
terrain_colors = [COLORS[4], COLORS[5], COLORS[6], COLORS[7]]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5))
fig2.suptitle(
    "ROM Comparison — Slope & Stair Conditions\n"
    "Right Leg (n=10), Individual subjects shown",
    fontsize=12, fontweight='bold')

for col_idx, jname in enumerate(JOINT_ORDER):
    ax        = axes2[col_idx]
    data_plot = [mode_rom[m][jname] for m in terrain_modes]

    bp = ax.boxplot(data_plot,
                    patch_artist=True,
                    widths=0.5,
                    medianprops=dict(color='black',
                                     linewidth=2.0))
    for patch, color in zip(bp['boxes'], terrain_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for element in ['whiskers','caps','fliers']:
        for item in bp[element]:
            item.set_color('#444444')
            item.set_linewidth(0.8)

    # Individual subject dots
    for xi, vals in enumerate(data_plot, start=1):
        jitter = np.random.uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jitter, vals,
                   color='black', s=18,
                   zorder=5, alpha=0.7)

    ax.set_xticks(range(1, len(terrain_modes)+1))
    ax.set_xticklabels(
        ["Slope\nAsc", "Slope\nDes",
         "Stair\nAsc", "Stair\nDes"],
        fontsize=9)
    ax.set_title(jname, fontsize=11, fontweight='bold')
    if col_idx == 0:
        ax.set_ylabel("ROM (°)", fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.axvline(2.5, color='gray', linestyle=':',
               linewidth=0.8, alpha=0.6)

from matplotlib.patches import Patch
leg2 = [
    Patch(facecolor=COLORS[4], edgecolor='black',
          alpha=0.85, label="Slope Ascent"),
    Patch(facecolor=COLORS[5], edgecolor='black',
          alpha=0.85, label="Slope Descent"),
    Patch(facecolor=COLORS[6], edgecolor='black',
          alpha=0.85, label="Stair Ascent"),
    Patch(facecolor=COLORS[7], edgecolor='black',
          alpha=0.85, label="Stair Descent"),
]
fig2.legend(handles=leg2,
            loc='lower center',
            bbox_to_anchor=(0.5, -0.04),
            ncol=4, fontsize=9,
            framealpha=0.9, edgecolor='#aaaaaa')
plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.subplots_adjust(wspace=0.28)
plt.savefig(
    os.path.join(OUTPUT_DIR, "fig_condition_comparison_slope_stair.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_condition_comparison_slope_stair.png")

# ── Significant pairs summary ─────────────────────────────────────────────────
print(f"\n── Significant Mode Pairs (p_bonf < 0.05) ───────────────────")
sig_pairs = cond_comp_df[cond_comp_df.Significant == "Yes"]
print(f"  Total significant pairs: {len(sig_pairs)} / "
      f"{len(cond_comp_df)}")
print(f"\n  {'Joint':<15} {'Mode A':<16} {'Mode B':<16} "
      f"{'Δ ROM':>8} {'p_bonf':>9}")
print("  " + "-"*70)
for _, row in sig_pairs.iterrows():
    print(f"  {row.Joint:<15} {row.Mode_A:<16} {row.Mode_B:<16} "
          f"{row.Mean_Diff:>+8.1f}° {row.p_bonf:>9.4f}")

# Cell 12 — Speed-ROM Correlation: Spearman correlation between walking speed and joint ROM

In [ ]:
# ── Spearman correlation — Speed vs ROM ───────────────────────────────────────
print("="*60)
print("SPEED-ROM CORRELATION — Spearman")
print("Treadmill and Overground, Right leg")
print("="*60)

corr_rows = []

for loco, spd_ms_map in [
        ("Treadmill",  TM_SPEEDS_MS),
        ("Overground", OG_SPEEDS_MS)]:

    for jR, jname in STAT_JOINTS_R.items():

        all_speeds, all_roms, pid_labels = [], [], []

        for plabel in P_LABELS:
            for spd, spd_val in spd_ms_map.items():
                pdata = get_participant_rom(
                    plabel, loco, spd, "R", "L")
                if pdata and jR in pdata["rom_R"]:
                    all_speeds.append(spd_val)
                    all_roms.append(pdata["rom_R"][jR])
                    pid_labels.append(plabel)

        if len(all_speeds) < 4:
            continue

        r, p = stats.spearmanr(all_speeds, all_roms)
        sig  = "*" if p < 0.05 else "ns"

        corr_rows.append({
            "Locomotion":     loco,
            "Joint":          jname,
            "Side":           "R",
            "Spearman_r":     round(r, 3),
            "p_value":        round(p, 4),
            "Significant":    "Yes" if p < 0.05 else "No",
            "N_observations": len(all_speeds),
        })
        print(f"  {loco:12s} | {jname:15s} | "
              f"ρ={r:+.3f} | p={p:.4f} {sig} | "
              f"n={len(all_speeds)}")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(
    os.path.join(OUTPUT_DIR, "speed_rom_correlation.csv"), index=False)
print(f"\n✅ Saved: speed_rom_correlation.csv")

# ── Speed-ROM scatter plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle(
    "Speed–ROM Relationship — Spearman Correlation\n"
    "Right Leg (per subject per speed, n=10)",
    fontsize=12, fontweight='bold')

for row_idx, (loco, spd_ms_map) in enumerate([
        ("Treadmill",  TM_SPEEDS_MS),
        ("Overground", OG_SPEEDS_MS)]):

    for col_idx, (jR, jname) in enumerate(STAT_JOINTS_R.items()):
        ax = axes[row_idx, col_idx]

        # Collect per-subject trajectories
        subj_data = {}
        for plabel in P_LABELS:
            spds, roms = [], []
            for spd, spd_val in spd_ms_map.items():
                pdata = get_participant_rom(
                    plabel, loco, spd, "R", "L")
                if pdata and jR in pdata["rom_R"]:
                    spds.append(spd_val)
                    roms.append(pdata["rom_R"][jR])
            if spds:
                subj_data[plabel] = (spds, roms)

        # Plot per-subject lines
        all_speeds_flat, all_roms_flat = [], []
        for i, (plabel, (spds, roms)) in \
                enumerate(subj_data.items()):
            p_idx = P_LABELS.index(plabel)
            ax.plot(spds, roms, 'o-',
                    color=P_COLORS[p_idx],
                    alpha=0.65, markersize=5,
                    linewidth=1.0,
                    label=plabel if col_idx == 0
                                 and row_idx == 0
                    else "")
            all_speeds_flat.extend(spds)
            all_roms_flat.extend(roms)

        # Population regression line
        if len(all_speeds_flat) >= 4:
            m, b = np.polyfit(
                all_speeds_flat, all_roms_flat, 1)
            x_line = np.linspace(
                min(all_speeds_flat),
                max(all_speeds_flat), 50)
            ax.plot(x_line, m*x_line + b,
                    'k--', linewidth=1.8,
                    alpha=0.85, zorder=6,
                    label='Trend')

            # Spearman result
            r, p = stats.spearmanr(
                all_speeds_flat, all_roms_flat)
            sig  = "*" if p < 0.05 else "ns"
            ax.set_title(
                f"{jname}\nρ={r:+.3f}, p={p:.4f} {sig}",
                fontsize=10, fontweight='bold')
        else:
            ax.set_title(jname, fontsize=10,
                         fontweight='bold')

        if col_idx == 0:
            ax.set_ylabel(
                f"{loco}\nROM (°)", fontsize=10)
        ax.set_xlabel("Speed (m/s)", fontsize=9)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

# Participant legend
handles, labels = axes[0, 0].get_legend_handles_labels()
# Keep only participant handles (not trend line)
p_handles = [h for h, l in zip(handles, labels)
             if l in P_LABELS]
p_labels  = [l for l in labels if l in P_LABELS]

fig.legend(p_handles, p_labels,
           loc='lower center',
           bbox_to_anchor=(0.5, -0.02),
           ncol=10, fontsize=8,
           framealpha=0.9,
           edgecolor='#aaaaaa',
           markerscale=1.2)

plt.tight_layout(rect=[0, 0.05, 1, 0.94])
plt.subplots_adjust(hspace=0.38, wspace=0.30)
plt.savefig(
    os.path.join(OUTPUT_DIR, "fig_speed_rom_correlation.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_speed_rom_correlation.png")

# ── Print correlation summary ─────────────────────────────────────────────────
print(f"\n── Correlation Summary ──────────────────────────────────────")
print(f"  {'Locomotion':<12} {'Joint':<15} "
      f"{'ρ':>7} {'p':>9} {'Sig':>5}")
print("  " + "-"*52)
for _, row in corr_df.iterrows():
    sig = "*" if row.Significant == "Yes" else "ns"
    print(f"  {row.Locomotion:<12} {row.Joint:<15} "
          f"{row.Spearman_r:>+7.3f} "
          f"{row.p_value:>9.4f} {sig:>5}")

# Cell 13 — Inter-subject Variability: Coefficient of Variation and PCA condition clustering

In [ ]:
# ── Coefficient of Variation ──────────────────────────────────────────────────
print("="*60)
print("INTER-SUBJECT VARIABILITY")
print("Coefficient of Variation (CV%) + PCA Condition Clustering")
print("="*60)

# Build CV from summary_df (right leg)
right_df_cv = summary_df[summary_df.Side == "R"].copy()
right_df_cv["CV_pct"] = (
    right_df_cv["ROM_SD"] /
    right_df_cv["ROM_Mean"] * 100).round(2)

# Group by mode — average CV across speeds/reps within mode
mode_cv_map = {
    "Treadmill":    [f"Treadmill {s}"  for s in SPEED_ORDER],
    "Overground":   [f"Overground {s}" for s in SPEED_ORDER],
    "OG Round":     ["Overground Round"],
    "OG Obstacles": ["Overground Obstacles"],
    "Slope Ascent": ["Slope Rep1 Ascent",  "Slope Rep2 Ascent"],
    "Slope Descent":["Slope Rep1 Descent", "Slope Rep2 Descent"],
    "Stair Ascent": ["Stair Rep1 Ascent",  "Stair Rep2 Ascent"],
    "Stair Descent":["Stair Rep1 Descent", "Stair Rep2 Descent"],
}

cv_rows = []
for mode, cond_labels in mode_cv_map.items():
    for jname in JOINT_ORDER:
        sub = right_df_cv[
            right_df_cv.Condition.isin(cond_labels) &
            (right_df_cv.Joint == jname)]
        if not sub.empty:
            cv_rows.append({
                "Mode":        mode,
                "Joint":       jname,
                "CV_mean_pct": round(sub["CV_pct"].mean(), 2),
                "ROM_mean":    round(sub["ROM_Mean"].mean(), 2),
                "ROM_sd":      round(sub["ROM_SD"].mean(),  2),
                "n_conds":     len(sub),
            })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(
    os.path.join(OUTPUT_DIR, "coefficient_of_variation.csv"),
    index=False)
print(f"✅ Saved: coefficient_of_variation.csv")

# ── Print CV table ────────────────────────────────────────────────────────────
print(f"\n  {'Mode':<16} | "
      f"{'Hip CV%':>9} {'Knee CV%':>9} {'Ankle CV%':>10}")
print("  " + "-"*50)
for mode in mode_cv_map.keys():
    sub  = cv_df[cv_df.Mode == mode]
    vals = []
    for jname in JOINT_ORDER:
        row = sub[sub.Joint == jname]
        vals.append(f"{row.iloc[0].CV_mean_pct:>9.1f}"
                    if not row.empty else f"{'N/A':>9}")
    print(f"  {mode:<16} | {vals[0]} {vals[1]} {vals[2]}")

# ── CV Heatmap ────────────────────────────────────────────────────────────────
cv_pivot = cv_df.pivot(
    index="Mode", columns="Joint", values="CV_mean_pct")
cv_pivot = cv_pivot[JOINT_ORDER].reindex(
    list(mode_cv_map.keys()))

fig1, ax1 = plt.subplots(figsize=(8, 6))
sns.heatmap(cv_pivot,
            annot=True, fmt=".1f",
            cmap="YlOrRd",
            linewidths=0.5,
            linecolor='white',
            annot_kws={"size": 10, "weight": "bold"},
            ax=ax1,
            cbar_kws={"label": "CV (%)", "shrink": 0.8})
ax1.set_title(
    "Inter-subject ROM Variability\n"
    "Coefficient of Variation (%, Right Leg)",
    fontsize=12, fontweight='bold', pad=12)
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_xticklabels(
    ax1.get_xticklabels(), fontsize=10)
ax1.set_yticklabels(
    ax1.get_yticklabels(), fontsize=10, rotation=0)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "fig_cv_heatmap.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_cv_heatmap.png")

# ── CV summary statistics ─────────────────────────────────────────────────────
print(f"\n── CV Summary ───────────────────────────────────────────────")
print(f"  Overall CV range : "
      f"{cv_df.CV_mean_pct.min():.1f}% – "
      f"{cv_df.CV_mean_pct.max():.1f}%")
print(f"  Overall CV mean  : "
      f"{cv_df.CV_mean_pct.mean():.1f}% ± "
      f"{cv_df.CV_mean_pct.std():.1f}%")
for jname in JOINT_ORDER:
    sub = cv_df[cv_df.Joint == jname]
    print(f"  {jname:<15} : "
          f"{sub.CV_mean_pct.min():.1f}% – "
          f"{sub.CV_mean_pct.max():.1f}% "
          f"(mean={sub.CV_mean_pct.mean():.1f}%)")

# ── PCA — condition clustering ────────────────────────────────────────────────
print(f"\n{'='*60}")
print("PCA — CONDITION CLUSTERING")
print("ROM feature space (Hip, Knee, Ankle — Right Leg)")
print("="*60)

# Build feature matrix: each row = one mode
pca_data, pca_labels = [], []
for mode in mode_cv_map.keys():
    sub      = cv_df[cv_df.Mode == mode]
    feats    = []
    for jname in JOINT_ORDER:
        row = sub[sub.Joint == jname]
        feats.append(
            row.iloc[0].ROM_mean if not row.empty
            else np.nan)
    if not any(np.isnan(feats)):
        pca_data.append(feats)
        pca_labels.append(mode)

pca_colors_map = {
    "Treadmill":    COLORS[0],
    "Overground":   COLORS[1],
    "OG Round":     COLORS[2],
    "OG Obstacles": COLORS[3],
    "Slope Ascent": COLORS[4],
    "Slope Descent":COLORS[5],
    "Stair Ascent": COLORS[6],
    "Stair Descent":COLORS[7],
}

if len(pca_data) >= 3:
    X      = StandardScaler().fit_transform(
        np.array(pca_data))
    pca    = PCA(n_components=2)
    coords = pca.fit_transform(X)
    var    = pca.explained_variance_ratio_ * 100

    print(f"  PC1: {var[0]:.1f}% variance explained")
    print(f"  PC2: {var[1]:.1f}% variance explained")
    print(f"  Total: {var[0]+var[1]:.1f}%")

    # Loadings
    print(f"\n  PCA Loadings:")
    print(f"  {'Joint':<15} {'PC1':>8} {'PC2':>8}")
    print("  " + "-"*32)
    for j, jname in enumerate(JOINT_ORDER):
        print(f"  {jname:<15} "
              f"{pca.components_[0,j]:>+8.3f} "
              f"{pca.components_[1,j]:>+8.3f}")

    fig2, ax2 = plt.subplots(figsize=(9, 7))

    for i, (label, coord) in enumerate(
            zip(pca_labels, coords)):
        color = pca_colors_map.get(label, 'gray')
        ax2.scatter(coord[0], coord[1],
                    color=color, s=220,
                    zorder=5,
                    edgecolors='black',
                    linewidth=0.8)
        ax2.annotate(
            label,
            (coord[0], coord[1]),
            textcoords="offset points",
            xytext=(10, 5),
            fontsize=9,
            color='#222222')

    # Draw lines connecting related conditions
    # Slope ascent-descent
    for pair in [
        ("Slope Ascent",  "Slope Descent"),
        ("Stair Ascent",  "Stair Descent"),
        ("Treadmill",     "Overground"),
        ("OG Round",      "OG Obstacles"),
    ]:
        if pair[0] in pca_labels and pair[1] in pca_labels:
            i0 = pca_labels.index(pair[0])
            i1 = pca_labels.index(pair[1])
            ax2.plot(
                [coords[i0,0], coords[i1,0]],
                [coords[i0,1], coords[i1,1]],
                'k--', linewidth=0.7, alpha=0.4)

    ax2.axhline(0, color='gray', linestyle='--',
                alpha=0.4, linewidth=0.8)
    ax2.axvline(0, color='gray', linestyle='--',
                alpha=0.4, linewidth=0.8)

    ax2.set_xlabel(
        f"PC1 ({var[0]:.1f}% variance)",
        fontsize=11)
    ax2.set_ylabel(
        f"PC2 ({var[1]:.1f}% variance)",
        fontsize=11)
    ax2.set_title(
        "PCA — Locomotion Condition Clustering\n"
        "ROM Feature Space (Hip, Knee, Ankle — Right Leg)",
        fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.25, linestyle='--')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    # Loadings arrows
    scale = 2.0
    for j, jname in enumerate(JOINT_ORDER):
        ax2.annotate(
            "",
            xy=(pca.components_[0,j] * scale,
                pca.components_[1,j] * scale),
            xytext=(0, 0),
            arrowprops=dict(
                arrowstyle="->",
                color="darkred",
                lw=1.5))
        ax2.text(
            pca.components_[0,j] * scale * 1.12,
            pca.components_[1,j] * scale * 1.12,
            jname, color='darkred',
            fontsize=8.5, fontweight='bold')

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR,
                     "fig_pca_condition_clustering.png"),
        dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: fig_pca_condition_clustering.png")
else:
    print("⚠️  Not enough modes for PCA")

# Cell 14 — Example Application: Locomotion mode classification using ROM features (KNN, LOSO-CV)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report)
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("EXAMPLE APPLICATION — LOCOMOTION MODE CLASSIFICATION")
print("Cycle-level | Random Forest | LOSO-CV")
print("="*60)

# ── Condition map ─────────────────────────────────────────────────────────────
# sk_R/sk_L must match keys returned by get_heel_strikes()
CLASSIFY_MODES = {
    "Treadmill":     ("Treadmill",  SPEED_ORDER,
                      "R",      "L"),
    "Overground":    ("Overground", SPEED_ORDER,
                      "R",      "L"),
    "OG Round":      ("Overground", ["round"],
                      "R",      "L"),
    "OG Obstacles":  ("Overground", ["obstacles"],
                      "R",      "L"),
    "Slope Ascent":  ("Slope", ["1","2"],
                      "asc_R",  "asc_L"),
    "Slope Descent": ("Slope", ["1","2"],
                      "desc_R", "desc_L"),
    "Stair Ascent":  ("Stair", ["1","2"],
                      "asc_R",  "asc_L"),
    "Stair Descent": ("Stair", ["1","2"],
                      "desc_R", "desc_L"),
}

# ── Feature extraction helpers ────────────────────────────────────────────────

def extract_signal_features(signal_1d):
    """6 statistical features from a 1D signal segment."""
    if len(signal_1d) < 3:
        return [0.0] * 6
    s = signal_1d.astype(float)
    return [
        float(np.mean(s)),
        float(np.std(s)),
        float(np.sqrt(np.mean(s**2))),  # RMS
        float(np.max(s)),
        float(np.min(s)),
        float(np.max(s) - np.min(s)),   # range
    ]


def get_cycle_windows(hs_indices, n_frames):
    """Return valid (start, end) frame pairs from heel strikes."""
    windows = []
    for i in range(len(hs_indices) - 1):
        start = int(hs_indices[i])
        end   = int(hs_indices[i + 1])
        dur   = end - start
        if 30 <= dur <= 300 and end < n_frames:
            windows.append((start, end))
    return windows


def extract_imu_features_per_cycle(imu_raw, windows):
    """
    IMU statistical features per gait cycle.
    imu_raw: (T x 48) — 8 sensors x 6 channels
    Returns: (n_cycles x 288)
    """
    feats = []
    for start, end in windows:
        seg      = imu_raw[start:end, :]
        cyc_feat = []
        for ch in range(48):
            cyc_feat.extend(
                extract_signal_features(seg[:, ch]))
        feats.append(cyc_feat)
    return np.array(feats) if feats else None


def extract_hof_features_per_cycle(hof_left, hof_right,
                                    windows):
    """
    HOF statistical features per gait cycle.
    HOF at 30 Hz, kinematics at 100 Hz → scale indices.
    Returns: (n_cycles x 108)
    """
    scale = 30.0 / 100.0
    feats = []
    for start, end in windows:
        hof_s    = int(start * scale)
        hof_e    = int(end   * scale)
        cyc_feat = []
        for hof in [hof_left, hof_right]:
            if hof is None:
                cyc_feat.extend([0.0] * 54)
                continue
            s_c = min(hof_s, len(hof) - 1)
            e_c = min(hof_e, len(hof))
            if e_c <= s_c:
                cyc_feat.extend([0.0] * 54)
                continue
            seg = hof[s_c:e_c, :]
            for f in range(18):
                # mean, std, rms only → 18x3=54 per camera
                cyc_feat.extend(
                    extract_signal_features(
                        seg[:, f])[:3])
        feats.append(cyc_feat)
    return np.array(feats) if feats else None


def extract_rom_features_per_cycle(kin_data, joint_names,
                                    windows):
    """
    Per-cycle ROM for hip, knee, ankle (R+L).
    Returns: (n_cycles x 6)
    """
    joints_rl = [
        "hip_flexion_r", "hip_flexion_l",
        "knee_angle_r",  "knee_angle_l",
        "ankle_angle_r", "ankle_angle_l",
    ]
    feats = []
    for start, end in windows:
        cyc_feat = []
        for jcol in joints_rl:
            if jcol in joint_names:
                idx = joint_names.index(jcol)
                seg = kin_data[start:end, idx]
                cyc_feat.append(float(np.ptp(seg)))
            else:
                cyc_feat.append(0.0)
        feats.append(cyc_feat)
    return np.array(feats) if feats else None


# ── Build per-cycle feature matrices ─────────────────────────────────────────
print("\nExtracting per-cycle features...")
print("Heel strikes detected on-the-fly from raw markers")
print("IMU: 288 features | HOF: 108 features | ROM: 6 features")
print("-"*60)

X_rom,  X_imu,  X_hof,  X_fusion = [], [], [], []
y_all,  subj_all                  = [], []

for mode_label, (loco, conds, sk_R, sk_L) in \
        CLASSIFY_MODES.items():

    for plabel in P_LABELS:
        mode_cycles = 0

        for cond in conds:

            # Step 1 — Detect heel strikes via Cell 4 functions
            hs_result = get_heel_strikes(plabel, loco, cond)
            if hs_result is None:
                continue
            if sk_R not in hs_result:
                continue
            hs_R_arr = hs_result[sk_R]
            if len(hs_R_arr) < 3:
                continue

            # Step 2 — Load kinematics
            kin_data, joint_names, _ = load_kinematics(
                plabel, loco, cond)
            if kin_data is None:
                continue
            n_frames = kin_data.shape[0]

            # Step 3 — Load IMU + HOF from HDF5
            base = (f"participants/{plabel}/"
                    f"{loco.lower()}/{cond}")
            with h5py.File(HDF5_PATH, "r") as hf:
                imu_raw   = None
                hof_left  = None
                hof_right = None
                if f"{base}/imu/raw" in hf:
                    imu_raw = hf[f"{base}/imu/raw"][:]
                if f"{base}/hof/left/features" in hf:
                    hof_left  = hf[
                        f"{base}/hof/left/features"][:]
                if f"{base}/hof/right/features" in hf:
                    hof_right = hf[
                        f"{base}/hof/right/features"][:]

            # Step 4 — Get cycle windows
            windows = get_cycle_windows(hs_R_arr, n_frames)
            if len(windows) < 2:
                continue

            # Step 5 — Extract features per cycle
            rom_feats = extract_rom_features_per_cycle(
                kin_data, joint_names, windows)
            imu_feats = (
                extract_imu_features_per_cycle(
                    imu_raw, windows)
                if imu_raw is not None else None)
            hof_feats = (
                extract_hof_features_per_cycle(
                    hof_left, hof_right, windows)
                if (hof_left is not None or
                    hof_right is not None) else None)

            if (rom_feats is None or
                imu_feats is None or
                hof_feats is None):
                continue

            min_cyc = min(len(rom_feats),
                          len(imu_feats),
                          len(hof_feats))
            if min_cyc < 2:
                continue

            X_rom.append(rom_feats[:min_cyc])
            X_imu.append(imu_feats[:min_cyc])
            X_hof.append(hof_feats[:min_cyc])
            X_fusion.append(np.hstack([
                imu_feats[:min_cyc],
                hof_feats[:min_cyc]]))
            y_all.extend([mode_label] * min_cyc)
            subj_all.extend([plabel]  * min_cyc)
            mode_cycles += min_cyc

        print(f"  ✅ {plabel} — {mode_label:<16} "
              f"({mode_cycles} cycles)")

# ── Verify before stacking ────────────────────────────────────────────────────
print(f"\n  Arrays collected: {len(X_rom)}")
if len(X_rom) == 0:
    raise ValueError(
        "❌ No cycles collected. "
        "Check get_heel_strikes() returns for your conditions.")

X_rom    = np.vstack(X_rom)
X_imu    = np.vstack(X_imu)
X_hof    = np.vstack(X_hof)
X_fusion = np.vstack(X_fusion)
y_all    = np.array(y_all)
subj_all = np.array(subj_all)

print(f"\n✅ Feature extraction complete")
print(f"  Total cycles  : {len(y_all)}")
print(f"  ROM features  : {X_rom.shape[1]}")
print(f"  IMU features  : {X_imu.shape[1]}")
print(f"  HOF features  : {X_hof.shape[1]}")
print(f"  Fusion feats  : {X_fusion.shape[1]}")
print(f"\n  Class distribution:")
for mode in sorted(set(y_all)):
    n = np.sum(y_all == mode)
    print(f"    {mode:<16}: {n:>5} cycles")

# ── LOSO-CV with Random Forest ────────────────────────────────────────────────
def run_loso_cv(X, y, subjects, experiment_name,
                n_estimators=200):
    """
    Leave-One-Subject-Out CV with Random Forest.
    Returns: overall_acc, per_class_acc, all_true,
             all_pred, subj_accs, mean_importances
    """
    print(f"\n── {experiment_name} ─────────────────────────")
    all_true, all_pred = [], []
    subj_accs          = {}
    importances        = []

    for test_subj in P_LABELS:
        train_mask = subjects != test_subj
        test_mask  = subjects == test_subj

        X_train = X[train_mask]
        y_train = y[train_mask]
        X_test  = X[test_mask]
        y_test  = y[test_mask]

        if len(X_test) == 0 or len(set(y_train)) < 2:
            continue

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            n_jobs=-1,
            random_state=42,
            max_features='sqrt',
            class_weight='balanced')
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        importances.append(clf.feature_importances_)
        acc = accuracy_score(y_test, y_pred)
        subj_accs[test_subj] = acc
        all_true.extend(y_test)
        all_pred.extend(y_pred)
        print(f"  {test_subj}: {acc:.3f} "
              f"({int(acc*len(y_test))}/{len(y_test)})")

    overall_acc  = accuracy_score(all_true, all_pred)
    class_labels = sorted(set(y))
    per_class    = {}
    for label in class_labels:
        mask    = np.array(all_true) == label
        correct = np.array(all_pred)[mask] == label
        per_class[label] = (float(correct.mean())
                            if correct.size > 0 else 0.0)

    mean_imp = (np.mean(importances, axis=0)
                if importances else None)

    accs = list(subj_accs.values())
    print(f"\n  Overall  : {overall_acc*100:.1f}%")
    print(f"  Mean±SD  : "
          f"{np.mean(accs)*100:.1f}±"
          f"{np.std(accs)*100:.1f}%")
    print(f"  Chance   : "
          f"{100/len(class_labels):.1f}%")

    return (overall_acc, per_class,
            all_true, all_pred,
            subj_accs, mean_imp)


# ── Run all 4 experiments ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("RUNNING 4 CLASSIFICATION EXPERIMENTS")
print("="*60)

exp_configs = {
    "ROM\n(Kinematic\nBaseline)":  X_rom,
    "IMU\n(Wearable)":             X_imu,
    "HOF\n(Wearable\nVision)":     X_hof,
    "IMU+HOF\n(Multimodal)":       X_fusion,
}

results = {}
for exp_name, X_exp in exp_configs.items():
    results[exp_name] = run_loso_cv(
        X_exp, y_all, subj_all,
        exp_name.replace('\n', ' '))

class_labels = sorted(set(y_all))
n_classes    = len(class_labels)
chance       = 1.0 / n_classes

# Color map for modes
pca_colors_map = {
    "Treadmill":    COLORS[0],
    "Overground":   COLORS[1],
    "OG Round":     COLORS[2],
    "OG Obstacles": COLORS[3],
    "Slope Ascent": COLORS[4],
    "Slope Descent":COLORS[5],
    "Stair Ascent": COLORS[6],
    "Stair Descent":COLORS[7],
}

# ── Figure 1: Accuracy comparison ─────────────────────────────────────────────
exp_names = list(results.keys())
exp_accs  = [results[e][0] for e in exp_names]

fig1, ax1 = plt.subplots(figsize=(10, 5))
bars = ax1.bar(range(len(exp_names)),
               exp_accs,
               color=COLORS[:4],
               edgecolor='black',
               linewidth=0.7,
               alpha=0.85,
               width=0.5)

for bar, acc in zip(bars, exp_accs):
    ax1.text(bar.get_x() + bar.get_width()/2,
             acc + 0.01,
             f"{acc*100:.1f}%",
             ha='center', va='bottom',
             fontsize=11, fontweight='bold')

ax1.axhline(chance,
            color='red', linestyle='--',
            linewidth=1.5, alpha=0.8,
            label=f"Chance ({chance*100:.1f}%)")

ax1.set_xticks(range(len(exp_names)))
ax1.set_xticklabels(exp_names, fontsize=9)
ax1.set_ylabel("Overall Accuracy (LOSO-CV)", fontsize=10)
ax1.set_ylim(0, 1.18)
ax1.set_title(
    "Locomotion Mode Classification — Random Forest\n"
    f"{n_classes} classes | Cycle-level LOSO-CV | n=10 subjects",
    fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR,
                 "fig_clf_accuracy_comparison.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_clf_accuracy_comparison.png")

# ── Figure 2: Confusion matrix — best experiment ──────────────────────────────
best_exp = max(results, key=lambda e: results[e][0])
_, _, best_true, best_pred, _, _ = results[best_exp]

cm      = confusion_matrix(best_true, best_pred,
                           labels=class_labels)
cm_norm = (cm.astype(float) /
           cm.sum(axis=1, keepdims=True))

fig2, axes2 = plt.subplots(1, 2, figsize=(18, 7))

for ax_idx, (data, fmt_, title_suffix, cbar_label) in \
        enumerate([
            (cm,      'd',    "Counts",     "Count"),
            (cm_norm, '.2f',  "Normalized", "Recall"),
        ]):
    ax = axes2[ax_idx]
    sns.heatmap(data,
                annot=True, fmt=fmt_,
                cmap='Blues',
                vmin=0,
                vmax=(None if fmt_=='d' else 1),
                xticklabels=class_labels,
                yticklabels=class_labels,
                linewidths=0.5,
                linecolor='white',
                annot_kws={"size": 9},
                ax=ax,
                cbar_kws={"shrink": 0.8,
                           "label": cbar_label})
    ax.set_title(
        f"Confusion Matrix ({title_suffix})\n"
        f"Best: {best_exp.replace(chr(10),' ')} — "
        f"Acc={results[best_exp][0]*100:.1f}%",
        fontsize=10, fontweight='bold')
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True",      fontsize=10)
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=30, ha='right', fontsize=8)
    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR,
                 "fig_clf_confusion_best.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Saved: fig_clf_confusion_best.png "
      f"(best: {best_exp.replace(chr(10),' ')})")

# ── Figure 3: Per-class accuracy — all 4 experiments ─────────────────────────
fig3, axes3 = plt.subplots(1, 4, figsize=(18, 5),
                            sharey=True)
fig3.suptitle(
    "Per-class Accuracy — All Experiments\n"
    "Random Forest LOSO-CV",
    fontsize=11, fontweight='bold')

for ax_idx, (exp_name, res) in \
        enumerate(results.items()):
    ax        = axes3[ax_idx]
    per_class = res[1]
    labels    = list(per_class.keys())
    accs      = [per_class[l] for l in labels]
    colors    = [pca_colors_map.get(l, 'gray')
                 for l in labels]

    bars = ax.bar(range(len(labels)), accs,
                  color=colors,
                  edgecolor='black',
                  linewidth=0.5,
                  alpha=0.85)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2,
                acc + 0.01,
                f"{acc*100:.0f}%",
                ha='center', va='bottom',
                fontsize=7, fontweight='bold')

    ax.axhline(chance,
               color='red', linestyle='--',
               linewidth=1.0, alpha=0.7,
               label=f"Chance")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels,
                       rotation=35, ha='right',
                       fontsize=7.5)
    ax.set_title(
        f"{exp_name.replace(chr(10),' ')}\n"
        f"Acc={res[0]*100:.1f}%",
        fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.2)
    if ax_idx == 0:
        ax.set_ylabel("Per-class Accuracy",
                      fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(
    os.path.join(OUTPUT_DIR,
                 "fig_clf_per_class.png"),
    dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_clf_per_class.png")

# ── Figure 4: IMU sensor importance ──────────────────────────────────────────
imu_imp = results["IMU\n(Wearable)"][5]

if imu_imp is not None:
    sensor_labels = [
        "Sternum", "Sacrum",
        "R.Thigh", "L.Thigh",
        "R.Shank", "L.Shank",
        "R.Foot",  "L.Foot",
    ]
    # Reshape: 8 sensors × 6 ch × 6 stats = 288
    imp_reshaped = imu_imp.reshape(8, 6, 6)
    sensor_imp   = imp_reshaped.sum(axis=(1, 2))
    sensor_imp  /= sensor_imp.sum()

    fig4, ax4 = plt.subplots(figsize=(9, 4))
    bars4 = ax4.bar(range(8),
                    sensor_imp,
                    color=COLORS[:8],
                    edgecolor='black',
                    linewidth=0.6,
                    alpha=0.85)
    for bar, imp in zip(bars4, sensor_imp):
        ax4.text(bar.get_x() + bar.get_width()/2,
                 imp + 0.002,
                 f"{imp*100:.1f}%",
                 ha='center', va='bottom',
                 fontsize=8.5, fontweight='bold')

    ax4.set_xticks(range(8))
    ax4.set_xticklabels(sensor_labels, fontsize=9)
    ax4.set_ylabel("Relative Importance",
                   fontsize=10)
    ax4.set_title(
        "IMU Sensor Importance — Random Forest\n"
        "Aggregated across channels and statistics",
        fontsize=11, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR,
                     "fig_imu_sensor_importance.png"),
        dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: fig_imu_sensor_importance.png")

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("CLASSIFICATION SUMMARY")
print("="*60)
print(f"  Classifier  : Random Forest (200 trees, balanced)")
print(f"  Validation  : Leave-One-Subject-Out CV")
print(f"  Granularity : Cycle-level ({len(y_all)} cycles)")
print(f"  Classes     : {n_classes} locomotion modes")
print(f"  Chance      : {chance*100:.1f}%")
print()
print(f"  {'Experiment':<28} {'Accuracy':>10}")
print("  " + "-"*40)
for exp_name, res in results.items():
    clean = exp_name.replace('\n', ' ')
    print(f"  {clean:<28} {res[0]*100:>9.1f}%")

best_clean = best_exp.replace('\n', ' ')
print(f"\n  Best : {best_clean} "
      f"({results[best_exp][0]*100:.1f}%)")

# ── Output file list ──────────────────────────────────────────────────────────
print(f"\n✅ All outputs saved to: {OUTPUT_DIR}")
all_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith('.png') or f.endswith('.csv')])
print(f"\n── Output files ({len(all_files)}) ──────────────")
for f in all_files:
    size = os.path.getsize(
        os.path.join(OUTPUT_DIR, f)) / 1024
    print(f"  {f:<48} {size:>7.1f} KB")

In [ ]:
# ── Save results for figure editing ──────────────────────────────────────────
import pickle

results_save = {
    "results":      results,
    "y_all":        y_all,
    "subj_all":     subj_all,
    "class_labels": class_labels,
    "n_classes":    n_classes,
    "chance":       chance,
    "best_exp":     best_exp,
    "best_true":    best_true,
    "best_pred":    best_pred,
    "pca_colors_map": pca_colors_map,
}

with open("/content/clf_results.pkl", "wb") as f:
    pickle.dump(results_save, f)

print("✅ Results saved to /content/clf_results.pkl")
print("   Load in Cell 15 without rerunning classification.")

In [ ]:
from matplotlib.gridspec import GridSpec
import pickle

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FONT_SCALE  = 1.3
GAP_SCALE   = 0.03
BAR_WIDTH   = 0.20
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

F_title    = 9  * FONT_SCALE
F_ylabel   = 8  * FONT_SCALE
F_xtick    = 8  * FONT_SCALE
F_bar_lbl  = 7  * FONT_SCALE
F_legend   = 7  * FONT_SCALE
F_panel    = 11 * FONT_SCALE
F_cm_annot = 9  * FONT_SCALE
F_cm_tick  = 8  * FONT_SCALE
F_cm_label = 9  * FONT_SCALE
F_cm_title = 9  * FONT_SCALE

_left    = 0.04
_gap     = GAP_SCALE
_bw      = BAR_WIDTH
_bh      = 0.36          # ← slightly shorter bars
_top_bot = 0.58          # ← top row higher
_bot_bot = 0.12          # ← bottom row higher, more room for xtick labels

bar_axes_pos = [
    [_left,              _top_bot, _bw, _bh],
    [_left + _bw + _gap, _top_bot, _bw, _bh],
    [_left,              _bot_bot, _bw, _bh],
    [_left + _bw + _gap, _bot_bot, _bw, _bh],
]

cm_left   = _left + (_bw + _gap) * 2 + 0.03
cm_ax_pos = [cm_left, 0.08, 0.97 - cm_left, 0.86]

with open("/content/clf_results.pkl", "rb") as f:
    saved = pickle.load(f)

results        = saved["results"]
y_all          = saved["y_all"]
class_labels   = saved["class_labels"]
n_classes      = saved["n_classes"]
chance         = saved["chance"]
best_exp       = saved["best_exp"]
best_true      = saved["best_true"]
best_pred      = saved["best_pred"]
pca_colors_map = saved["pca_colors_map"]

print("✅ Results loaded")

PAPER_DIR = os.path.join(OUTPUT_DIR, "paper_figures")
os.makedirs(PAPER_DIR, exist_ok=True)

short_labels = {
    "OG Obstacles": "OG\nObs",
    "OG Round":     "OG\nRound",
    "Overground":   "OG",
    "Slope Ascent": "Slope\nAsc",
    "Slope Descent":"Slope\nDes",
    "Stair Ascent": "Stair\nAsc",
    "Stair Descent":"Stair\nDes",
    "Treadmill":    "TM",
}
tick_labels = [short_labels.get(l, l) for l in class_labels]

cm      = confusion_matrix(best_true, best_pred,
                           labels=class_labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig = plt.figure(figsize=(18, 10))  # ← taller figure

for ax_idx, (pos, (exp_name, res)) in enumerate(
        zip(bar_axes_pos, results.items())):

    ax        = fig.add_axes(pos)
    per_class = res[1]
    labels    = list(per_class.keys())
    accs      = [per_class[l] for l in labels]
    colors    = [pca_colors_map.get(l, 'gray') for l in labels]
    tick_lbls = [short_labels.get(l, l) for l in labels]

    ax.bar(range(len(labels)), accs,
           color=colors,
           edgecolor='black',
           linewidth=0.5,
           alpha=0.85,
           width=0.55)

    for i, acc in enumerate(accs):
        ax.text(i, acc + 0.012,
                f"{acc*100:.0f}%",
                ha='center', va='bottom',
                fontsize=F_bar_lbl,
                fontweight='bold')

    ax.axhline(chance,
               color='red', linestyle='--',
               linewidth=0.9, alpha=0.75,
               label="Chance (12.5%)")

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(tick_lbls,
                       rotation=45,       # ← was 30, more angle
                       ha='right',
                       fontsize=F_xtick,
                       rotation_mode='anchor')  # ← anchor prevents drift
    clean = exp_name.replace('\n', ' ')
    ax.set_title(f"{clean}\nAcc={res[0]*100:.1f}%",
                 fontsize=F_title, fontweight='bold')
    ax.set_ylim(0, 1.28)
    ax.set_xlim(-0.5, len(labels) - 0.5)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=F_xtick)
    ax.tick_params(axis='x', pad=2)      # ← tighter x tick padding

    if ax_idx in [0, 2]:
        ax.set_ylabel("Per-class Accuracy",
                      fontsize=F_ylabel)
    if ax_idx == 0:
        ax.legend(fontsize=F_legend,
                  loc='upper left',
                  framealpha=0.85)

ax_cm = fig.add_axes(cm_ax_pos)

sns.heatmap(cm_norm,
            annot=True, fmt='.2f',
            cmap='Blues',
            vmin=0, vmax=1,
            xticklabels=tick_labels,
            yticklabels=tick_labels,
            linewidths=0.5,
            linecolor='white',
            annot_kws={"size": F_cm_annot},
            ax=ax_cm,
            square=True,
            cbar_kws={"shrink": 0.5,
                      "pad": 0.02})

ax_cm.set_xlabel("Predicted", fontsize=F_cm_label)
ax_cm.set_ylabel("True",      fontsize=F_cm_label)
ax_cm.set_xticklabels(
    ax_cm.get_xticklabels(),
    rotation=30, ha='right',
    fontsize=F_cm_tick,
    rotation_mode='anchor')   # ← anchor for CM too
ax_cm.set_yticklabels(
    ax_cm.get_yticklabels(),
    rotation=0, fontsize=F_cm_tick)
ax_cm.set_title(
    f"Confusion Matrix — IMU+HOF Multimodal\n"
    f"(Acc={results[best_exp][0]*100:.1f}%)",
    fontsize=F_cm_title, fontweight='bold', pad=12)

fig.text(0.01, 0.97, "(A)", fontsize=F_panel,
         fontweight='bold', va='top')
fig.text(cm_left - 0.02, 0.97, "(B)",
         fontsize=F_panel,
         fontweight='bold', va='top')

path_final = os.path.join(PAPER_DIR, "fig_classification.png")
plt.savefig(path_final, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Saved: fig_classification.png")
print(f"   FONT_SCALE={FONT_SCALE} | BAR_WIDTH={BAR_WIDTH} | "
      f"GAP_SCALE={GAP_SCALE}")

In [ ]:
# ── Cell 15: Figure editing — load saved results ──────────────────────────────
import pickle

with open("/content/clf_results.pkl", "rb") as f:
    saved = pickle.load(f)

results       = saved["results"]
y_all         = saved["y_all"]
subj_all      = saved["subj_all"]
class_labels  = saved["class_labels"]
n_classes     = saved["n_classes"]
chance        = saved["chance"]
best_exp      = saved["best_exp"]
best_true     = saved["best_true"]
best_pred     = saved["best_pred"]
pca_colors_map = saved["pca_colors_map"]

print("✅ Results loaded successfully")
print(f"  Classes     : {class_labels}")
print(f"  Best exp    : {best_exp.replace(chr(10),' ')}")
print(f"  Best acc    : {results[best_exp][0]*100:.1f}%")
print(f"  Total cycles: {len(y_all)}")

In [ ]:
import shutil
from google.colab import files

# ── Zip the output folder ─────────────────────────────────────────────────────
zip_path = "/content/gait_outputs_archive"

shutil.make_archive(
    zip_path,      # output path (no .zip extension)
    'zip',         # format
    "/content",    # root directory
    "gait_outputs" # directory to zip
)

print(f"✅ Zipped: {zip_path}.zip")

# ── Check size ────────────────────────────────────────────────────────────────
size_mb = os.path.getsize(f"{zip_path}.zip") / (1024**2)
print(f"   Size: {size_mb:.1f} MB")

# ── Download ──────────────────────────────────────────────────────────────────
files.download(f"{zip_path}.zip")
print("✅ Download started — check your browser downloads")